# Exercise 7: Build a Personal Voice Assistant

An always-on assistant called **Loop**. You say *"Loop, book a dentist appointment tomorrow at 3 PM in Chicago"*; it books it, answers out loud, and remembers things about you. A web page shows what it heard, what it did and what it remembers.

```
mic -> speech to text -> wake word -> agent (tools + memory) -> text to speech -> speaker
                                          |
                                  appointments (SQLite), memory (Qdrant)
```

| Piece | Choice |
|---|---|
| Chat model | Ollama, any model with tool calling (`gemma4:12b-mlx` by default, `qwen3:8b` to compare) |
| Embeddings | Ollama `nomic-embed-text` |
| Agent | a small streaming tool-calling loop over `ollama.Client.chat` (no framework, so every tool call is visible) |
| Appointments | SQLite |
| Memory | Qdrant (server from `Week6/docker-compose.yml`, or embedded) |
| Microphone, beep | `sounddevice` + `numpy` |
| Speech to text | `faster-whisper` (or `mlx-whisper` on Apple Silicon) |
| Text to speech | macOS `say`, `espeak-ng` on Linux, `pyttsx3` on Windows, each as a child process so it can be killed |
| Web page | Streamlit, listener in a background thread created once with `st.cache_resource` |

**How this notebook works.** The deliverable is the project folder `voice-assistant/`. Each stage writes its files with `%%writefile` and then runs that stage's check. Do not move on until the check passes.

**Laptop or Colab.** The notebook detects Colab, as Exercise 6.1 does. Locally it uses Ollama and Qdrant on your machine and the real microphone and speakers. On Colab (pick a **GPU runtime**):
- the project folder lives on Drive (`MyDrive/ADS-NLP/Week7/voice-assistant`), so it survives disconnects;
- Ollama is installed in the runtime, started, and the models are pulled. The `colab:` section of `config.yaml` swaps in Linux model tags (`qwen3:8b`, `llama3.1:8b`; MLX models only run on Macs) and an embedded Qdrant store on Drive;
- the Colab server has no microphone or speaker, so audio goes through your browser: commands are recorded as short clips (push to talk; allow mic access when the browser asks), and replies play in the cell output. `stop()` removes the player, which silences it.

What Colab cannot do is the *always-on* part: continuous listening, and muting the mic while it speaks. Stages 0, 1, 2, 4 and 6 run fully on Colab; Stage 3 runs as push-to-talk; Stage 5's page runs typing-only. For the demo, run Stages 3 and 5 on a laptop.

The checks are also in `checks.py`, so you can run them from a terminal: `python checks.py stage1`.

In [1]:
import importlib, importlib.util, os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REQS = {"ollama": "ollama", "qdrant_client": "qdrant-client", "yaml": "pyyaml", "faster_whisper": "faster-whisper",
        "sounddevice": "sounddevice", "streamlit": "streamlit", "pandas": "pandas"}
missing = [pip for mod, pip in REQS.items() if importlib.util.find_spec(mod) is None]
if missing:   # e.g. a fresh Colab runtime
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
if IN_COLAB and shutil.which("espeak-ng") is None:   # the voice, and PortAudio so sounddevice imports
    subprocess.run("apt-get -qq update && apt-get -qq install -y espeak-ng libportaudio2 zstd > /dev/null", shell=True, check=True)

if IN_COLAB:   # keep the project on Drive, like Exercise 6.1
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/ADS-NLP/Week7/voice-assistant")
else:
    PROJECT = Path("voice-assistant")
if Path.cwd().resolve() != PROJECT.resolve():   # run every %%writefile inside the project folder
    PROJECT.mkdir(parents=True, exist_ok=True)
    os.chdir(PROJECT)
Path("va").mkdir(exist_ok=True)
print(f"Colab: {IN_COLAB}")
sys.path.insert(0, str(Path.cwd()))
print("Project folder:", Path.cwd())

def fresh(*names):
    # Re-import the project modules after a %%writefile, so the newest code runs.
    for mod in [m for m in sys.modules if m == "va" or m.startswith("va.") or m == "checks"]:
        del sys.modules[mod]
    mods = [importlib.import_module(n) for n in names]
    return mods[0] if len(mods) == 1 else mods

Mounted at /content/drive
Colab: True
Project folder: /content/drive/MyDrive/ADS-NLP/Week7/voice-assistant


---
## Stage 0: Setup (10 minutes)

1. Install Ollama and pull a tool-calling chat model, a second model for Stage 6, and an embedding model.
2. Start Qdrant (`docker compose up -d` in `Week6/`), or set `memory.qdrant_path` in the config to use an embedded store. The embedded store can only be opened by one process at a time, so do not run the notebook checks and the Streamlit page together with it.
3. Allow microphone access for the program that runs this notebook (on macOS: System Settings, Privacy & Security, Microphone, then tick Terminal, VS Code or whichever app runs Jupyter).

```bash
ollama pull gemma4:12b-mlx
ollama pull qwen3:8b
ollama pull nomic-embed-text
cd Week6 && docker compose up -d
```

**One config file.** Models, wake word, thresholds and paths are all in `config.yaml`. Nothing below hard-codes them. Paths are relative to the config file.

In [2]:
%%writefile config.yaml
# All settings for the voice assistant. Paths are relative to this file.

assistant:
  name: "Loop"                  # what the assistant calls itself
  timezone: null                # null = the laptop's local time zone
  follow_up_seconds: 8          # after a reply, accept speech without the wake word for this long
  history_turns: 8              # how many past user turns the agent sees

llm:
  host: "http://localhost:11434"
  model: "gemma4:12b-mlx"       # any Ollama chat model with tool calling
  compare_models:               # Stage 6: the eight checks are run on each of these
    - "gemma4:12b-mlx"
    - "qwen3:8b"
  temperature: 0.0
  keep_alive: "60m"             # keep the model loaded between requests
  think: false                  # for thinking models (qwen3, gpt-oss); null = do not send the flag
  max_tool_rounds: 6
  guards: true                  # code-level checks on tool calls (see Stage 1)

embeddings:
  model: "nomic-embed-text"
  query_prefix: "search_query: "        # nomic-embed-text expects these prefixes
  document_prefix: "search_document: "

appointments:
  db_path: "data/appointments.db"

memory:
  qdrant_url: "http://localhost:6333"   # docker compose up -d (Week6/docker-compose.yml)
  qdrant_path: null                     # or a folder, e.g. "data/qdrant", for embedded Qdrant (one process only)
  collection: "voice_assistant_memory"
  top_k: 4
  min_score: 0.45                       # ignore matches below this cosine similarity
  duplicate_score: 0.92                 # do not store a fact this close to an existing one
  extract_in_background: true

wake:
  word: "loop"
  # Spellings the speech-to-text model produced for "Loop". Add more from logs/assistant.log.
  variants: ["loop", "loops", "loop's", "looped", "lupe", "loupe", "loup", "luup", "lup", "lube", "lou", "lu"]
  fillers: ["ok", "okay", "hey", "hi", "hello", "so", "um", "uh", "oh", "yo", "alright", "please"]

audio:
  enabled: true                 # false = typing only (no microphone)
  sample_rate: 16000
  frame_ms: 30
  calibration_seconds: 1.0      # measure the room's noise level for this long at start-up
  threshold_factor: 3.0         # speech = louder than noise level x this
  min_threshold: 0.006          # never use a threshold below this RMS
  preroll_seconds: 0.35         # audio kept from before speech starts
  end_silence_seconds: 0.6      # quiet this long ends an utterance
  min_speech_seconds: 0.3       # shorter blips are dropped
  max_utterance_seconds: 20
  unmute_delay_seconds: 0.35    # keep the mic muted this long after speech ends (room echo)
  input_device: null            # null = default microphone
  start_tone: true              # short beep when it starts listening again

stt:
  backend: "faster-whisper"     # faster-whisper | mlx-whisper
  model: "small.en"             # faster-whisper size, or an mlx repo such as "mlx-community/whisper-small.en-mlx"
  compute_type: "int8"
  initial_prompt: "Loop, book an appointment. Loop, what is on my calendar?"

tts:
  backend: "auto"               # auto | say | espeak | pyttsx3
  voice: null                   # e.g. "Samantha" for macOS say
  rate: 190                     # words per minute

logging:
  path: "logs/assistant.log"
  level: "INFO"

# Applied on top of the settings above when running on Google Colab (detected automatically).
# Colab has no microphone, speaker, Docker or Apple GPU, so: Linux model tags, embedded Qdrant,
# clips recorded in the browser, and speech played in the notebook.
colab:
  llm.model: "qwen3:8b"
  llm.compare_models: ["qwen3:8b", "llama3.1:8b"]
  memory.qdrant_path: "data/qdrant"
  audio.enabled: false
  audio.calibration_seconds: 0.3     # each browser clip starts with a short quiet moment
  stt.compute_type: "float16"        # GPU runtime
  tts.backend: "notebook"

Overwriting config.yaml


In [3]:
%%writefile requirements.txt
# Python 3.10+
ollama>=0.5            # chat + embeddings, with tool calling
qdrant-client>=1.9     # memory store
pyyaml
numpy
sounddevice            # microphone and beep
faster-whisper         # speech to text (all platforms)
# mlx-whisper          # speech to text, much faster on Apple Silicon (set stt.backend: mlx-whisper)
pyttsx3; platform_system == "Windows"   # text to speech on Windows (macOS uses `say`, Linux `espeak-ng`)
streamlit>=1.37        # web page (needs st.fragment)
pandas

Overwriting requirements.txt


In [4]:
%%writefile .gitignore
data/
logs/
__pycache__/

Overwriting .gitignore


The README holds the Stage 3 failures and the Stage 6 table, which later cells fill in. So it is written only if it does not exist yet; rerunning the notebook never overwrites your results.

In [5]:
README = '# Loop: a local voice assistant (Exercise 7)\n\nSay "Loop, book a dentist appointment tomorrow at 3 PM in Chicago". It books it, answers out loud, and remembers things about you. Everything runs on your laptop.\n\n```\nmic -> segmenter (volume) -> faster-whisper -> wake word -> agent (Ollama, tools + memory) -> sentence splitter -> say / espeak\n                                                              |\n                                              SQLite appointments, Qdrant memory\n```\n\nThe notebook [`../Exercise-7-Voice-Assistant.ipynb`](../Exercise-7-Voice-Assistant.ipynb) builds this folder stage by stage and runs each check.\n\n## Setup\n\n```bash\nollama pull gemma4:12b-mlx      # or any chat model with tool calling; set llm.model in config.yaml\nollama pull qwen3:8b            # second model for Stage 6\nollama pull nomic-embed-text\ndocker compose -f ../../Week6/docker-compose.yml up -d     # Qdrant on :6333 (or set memory.qdrant_path)\npip install -r requirements.txt\n```\n\nLinux also needs `sudo apt install espeak-ng portaudio19-dev`. On macOS, allow microphone access for the Terminal (or VS Code / Jupyter) app under System Settings, Privacy & Security, Microphone.\n\n### On Google Colab\n\nOpen the notebook on a GPU runtime and run it top to bottom. It detects Colab (like Exercise 6.1): the project goes to `MyDrive/ADS-NLP/Week7/voice-assistant`, Ollama is installed and the models are pulled in the runtime, and the `colab:` section of `config.yaml` switches to `qwen3:8b` / `llama3.1:8b`, embedded Qdrant on Drive, and browser audio (`va/colab.py` records clips with the browser\'s mic; replies play in the cell output). Colab cannot listen continuously, so Stage 3 runs as push-to-talk and the Stage 5 page is typing-only. Do the voice demo on a laptop.\n\n## Run\n\n```bash\nstreamlit run app.py        # web page + always-on listening\npython cli.py --voice       # the same, in the terminal\npython cli.py               # type commands, hear replies\npython checks.py stage1     # the eight typed checks (also stage2, stage4, stage6)\n```\n\nAll settings (models, wake word and its spellings, audio thresholds, paths) are in `config.yaml`. Logs go to `logs/assistant.log`; every transcript is logged there, which is where new wake-word spellings come from.\n\n## Files\n\n| File | What it does |\n|---|---|\n| `config.yaml` | every setting |\n| `va/appointments.py` | SQLite store behind the four appointment tools |\n| `va/agent.py` | system prompt, tool schemas, the streaming tool-calling loop, guards |\n| `va/speech_out.py` | markdown clean-up, streaming sentence splitter, `Speaker` with instant `stop()` |\n| `va/speech_in.py` | noise-calibrated segmenter, microphone, faster-whisper / mlx-whisper |\n| `va/wake.py` | loose wake-word matching |\n| `va/memory.py` | Qdrant memory with Ollama embeddings, background fact extraction |\n| `va/colab.py` | Colab only: record a clip with the browser\'s mic, push-to-talk turn |\n| `va/assistant.py` | ties it together: loader, listener and worker threads, state for the page |\n| `app.py` | Streamlit page |\n| `cli.py` | terminal front end |\n| `checks.py` | the checks for Stages 1, 2, 4 and 6 |\n\n## Design notes\n\n- **Agent.** A small loop over `ollama.Client.chat(..., tools=TOOLS, stream=True)` instead of a framework, so any Ollama tool model works and the page can show each tool call. Every request rebuilds the system prompt with the current date and a list of the next seven dates, so "Friday" maps to a real date without the model doing calendar arithmetic.\n- **Guards in code, not only in the prompt** (`llm.guards`). `create_appointment` refuses a location the user never said; `delete_appointment` refuses unless the user\'s latest message is a yes to a question; an identical booking in the same conversation returns the existing id. When a guard refuses, the tool returns `ok: false` with an instruction, and the model asks the user.\n- **Speech.** Reply text streams into a sentence splitter; each finished sentence goes to a TTS child process right away. `stop()` kills the process. The mic is closed while thinking and speaking (plus 0.35 s for echo) and the audio is thrown away.\n- **Wake word.** The wake word counts only at the start of a sentence (after optional fillers like "okay" or "hey") or as a vocative between commas. Whisper gets an `initial_prompt` containing "Loop", which makes it spell the name correctly much more often.\n- **Memory.** Searched before each request (top 4 above a similarity threshold). Facts are extracted after the reply has been spoken, in a background thread, and only when the user said something about themselves. Anything that looks like a booking is filtered out.\n\n## Stage 3: spoken commands that failed\n\nCopy lines from `logs/assistant.log` (`heard ...`) for every spoken command that did not work.\n\n| # | What I said | What speech-to-text produced | What went wrong |\n|---|---|---|---|\n| | | | |\n\n## Stage 6: measurements\n\nFilled in by `python checks.py stage6` (or the last section of the notebook) from real runs.\n\n<!-- stage6 -->\n| Model | Checks passed (of 8) | Total seconds for the eight checks |\n|---|---|---|\n| (your default model) | | |\n| (one other model) | | |\n<!-- /stage6 -->\n\n**Biggest source of mistakes:** _(fill in after your runs, e.g. "the small model invented a location for the haircut and deleted without asking")_\n\n**What I changed to fix it:** _(e.g. "moved those rules from the prompt into the tools, which now refuse and tell the model to ask")_\n'

if not Path("README.md").exists():
    Path("README.md").write_text(README)
    print("README.md written")
else:
    print("README.md already exists; left as it is")

README.md already exists; left as it is


In [6]:
# %pip install -r requirements.txt

In [7]:
%%writefile va/__init__.py
"""Loop: a local, always-on voice assistant (Exercise 7)."""

Overwriting va/__init__.py


In [8]:
%%writefile va/config.py
"""Load config.yaml and set up logging. Every setting lives in the config file."""
from __future__ import annotations

import copy
import logging
import os
import sys
import logging.handlers
from pathlib import Path

import yaml

PROJECT_DIR = Path(__file__).resolve().parent.parent
DEFAULT_CONFIG = PROJECT_DIR / "config.yaml"


def on_colab() -> bool:
    return "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ


def load_config(path: str | Path | None = None, **overrides) -> dict:
    """Read the YAML config. `overrides` are dotted keys, e.g. {"llm.model": "qwen3:8b"}.

    On Google Colab the `colab:` section of the config is applied first.
    """
    path = Path(path or DEFAULT_CONFIG).resolve()
    cfg = yaml.safe_load(path.read_text())
    cfg["_dir"] = str(path.parent)
    cfg["_colab"] = on_colab()
    if cfg["_colab"]:
        for dotted, value in (cfg.get("colab") or {}).items():
            set_key(cfg, dotted, value)
    for dotted, value in overrides.items():
        set_key(cfg, dotted, value)
    return cfg


def set_key(cfg: dict, dotted: str, value) -> None:
    node = cfg
    *parents, last = dotted.split(".")
    for key in parents:
        node = node.setdefault(key, {})
    node[last] = value


def with_overrides(cfg: dict, **overrides) -> dict:
    """A copy of cfg with dotted-key overrides applied."""
    new = copy.deepcopy(cfg)
    for dotted, value in overrides.items():
        set_key(new, dotted, value)
    return new


def resolve_path(cfg: dict, relative: str | None) -> Path | None:
    """Paths in the config are relative to the config file's folder."""
    if relative is None:
        return None
    p = Path(relative)
    return p if p.is_absolute() else Path(cfg["_dir"]) / p


def setup_logging(cfg: dict) -> logging.Logger:
    """Rotating log file plus console. Safe to call more than once."""
    log = logging.getLogger("va")
    if getattr(log, "_va_configured", False):
        return log
    level = getattr(logging, str(cfg["logging"].get("level", "INFO")).upper(), logging.INFO)
    log.setLevel(level)
    fmt = logging.Formatter("%(asctime)s %(levelname)-7s %(threadName)-10s %(message)s")
    path = resolve_path(cfg, cfg["logging"]["path"])
    path.parent.mkdir(parents=True, exist_ok=True)
    fh = logging.handlers.RotatingFileHandler(path, maxBytes=1_000_000, backupCount=3)
    fh.setFormatter(fmt)
    log.addHandler(fh)
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    sh.setLevel(logging.WARNING)
    log.addHandler(sh)
    log._va_configured = True
    return log

Overwriting va/config.py


### Start Ollama and pull the models

Locally this checks that Ollama is running and lists any models you still need to pull. On Colab it installs Ollama, starts `ollama serve` in the background and pulls every model named in the config (the chat model, the Stage 6 models and the embedding model). The first time takes about 5 minutes on a GPU runtime.

In [9]:
config = fresh("va.config")
cfg = config.load_config()
print("chat model:", cfg["llm"]["model"], "| compare:", cfg["llm"]["compare_models"], "| embeddings:", cfg["embeddings"]["model"])

def ollama_up():
    try:
        urllib.request.urlopen(cfg["llm"]["host"] + "/api/tags", timeout=3)
        return True
    except Exception:
        return False

if IN_COLAB and shutil.which("ollama") is None:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True, stdout=subprocess.DEVNULL)
if not ollama_up() and shutil.which("ollama"):
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if ollama_up():
            break
        time.sleep(1)
assert ollama_up(), "Ollama is not running: start the Ollama app (or `ollama serve`)"

import ollama
wanted = list(dict.fromkeys([cfg["llm"]["model"], *cfg["llm"]["compare_models"], cfg["embeddings"]["model"]]))
have = {m.model for m in ollama.Client(host=cfg["llm"]["host"]).list().models}
for name in wanted:
    if name in have or f"{name}:latest" in have:
        print("have   ", name)
    elif IN_COLAB:
        print("pulling", name)
        subprocess.run(["ollama", "pull", name], check=True, stdout=subprocess.DEVNULL)
    else:
        print("MISSING", name, "-> run: ollama pull", name)

chat model: qwen3:8b | compare: ['qwen3:8b', 'llama3.1:8b'] | embeddings: nomic-embed-text
pulling qwen3:8b
pulling llama3.1:8b
pulling nomic-embed-text


### Check 0: the models answer, the stores accept a write and a read

In [10]:
import sqlite3, uuid, ollama, numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

config = fresh("va.config")
cfg = config.load_config()
client = ollama.Client(host=cfg["llm"]["host"])

# 1) The chat model answers, and it calls a tool when it should.
weather_tool = {"type": "function", "function": {"name": "get_weather", "description": "Weather for a city",
                "parameters": {"type": "object", "required": ["city"], "properties": {"city": {"type": "string"}}}}}
r = client.chat(model=cfg["llm"]["model"], messages=[{"role": "user", "content": "What is the weather in Chicago?"}],
                tools=[weather_tool], keep_alive=cfg["llm"]["keep_alive"])
calls = [(t.function.name, t.function.arguments) for t in (r.message.tool_calls or [])]
print("chat model tool call:", calls)
assert calls and calls[0][0] == "get_weather", "this model did not call the tool; pick another model"

# 2) The embedding model answers.
vec = client.embed(model=cfg["embeddings"]["model"], input=["search_document: hello"]).embeddings[0]
print("embedding size:", len(vec))

# 3) SQLite: write and read.
db = config.resolve_path(cfg, cfg["appointments"]["db_path"]); db.parent.mkdir(parents=True, exist_ok=True)
with sqlite3.connect(db) as con:
    con.execute("CREATE TABLE IF NOT EXISTS smoke (x TEXT)"); con.execute("INSERT INTO smoke VALUES ('ok')")
    print("sqlite:", con.execute("SELECT x FROM smoke").fetchone()); con.execute("DROP TABLE smoke")

# 4) Qdrant: write and read.
m = cfg["memory"]
q = QdrantClient(path=str(config.resolve_path(cfg, m["qdrant_path"]))) if m["qdrant_path"] else QdrantClient(url=m["qdrant_url"])
if q.collection_exists("smoke_test"):
    q.delete_collection("smoke_test")
q.create_collection("smoke_test", vectors_config=VectorParams(size=len(vec), distance=Distance.COSINE))
q.upsert("smoke_test", points=[PointStruct(id=str(uuid.uuid4()), vector=vec, payload={"text": "hello"})])
print("qdrant:", q.query_points("smoke_test", query=vec, limit=1).points[0].payload)
q.delete_collection("smoke_test"); q.close()

chat model tool call: [('get_weather', {'city': 'Chicago'})]
embedding size: 768
sqlite: ('ok',)
qdrant: {'text': 'hello'}


In [13]:
%%writefile va/colab.py
"""Google Colab helpers. Colab runs on a server with no microphone or speaker, so audio goes
through the browser: record a clip with JavaScript, decode it with ffmpeg, then hand it to the
same segmenter, speech to text and wake word as on a laptop. Replies play in the cell output
(tts.backend: notebook).
"""
from __future__ import annotations

import base64
import subprocess
import time

import numpy as np

_RECORD_JS = """
async function vaRecord(ms) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream), chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  const note = document.createElement('div');
  note.style = 'font: 600 18px sans-serif; color: white; background: #16a34a; padding: 8px 14px; border-radius: 8px; display: inline-block';
  document.body.appendChild(note);
  rec.start();
  const t0 = Date.now();
  const timer = setInterval(() => { note.textContent = '🎙️ Listening… ' + Math.max(0, (ms - (Date.now() - t0)) / 1000).toFixed(1) + ' s'; }, 100);
  await new Promise(r => setTimeout(r, ms));
  rec.stop();
  await new Promise(r => rec.onstop = r);
  clearInterval(timer); note.remove();
  stream.getTracks().forEach(t => t.stop());
  const bytes = new Uint8Array(await new Blob(chunks).arrayBuffer());
  let s = ''; for (let i = 0; i < bytes.length; i += 8192) s += String.fromCharCode.apply(null, bytes.subarray(i, i + 8192));
  return btoa(s);
}
"""


def record_clip(seconds: float = 6.0, sample_rate: int = 16000) -> np.ndarray:
    """Record from the browser's microphone. Allow microphone access when the browser asks."""
    from google.colab.output import eval_js
    from IPython.display import Javascript, display
    display(Javascript(_RECORD_JS))
    data = base64.b64decode(eval_js(f"vaRecord({int(seconds * 1000)})"))
    out = subprocess.run(["ffmpeg", "-loglevel", "error", "-i", "pipe:0", "-f", "f32le", "-ac", "1",
                          "-ar", str(sample_rate), "pipe:1"], input=data, capture_output=True, check=True)
    return np.frombuffer(out.stdout, dtype=np.float32).copy()


def listen_once(assistant, seconds: float = 6.0) -> dict:
    """One push-to-talk turn: record, cut out the speech, transcribe, wake word, then wait for the reply."""
    from .speech_in import Segmenter
    started = time.monotonic()
    clip = record_clip(seconds, assistant.cfg["audio"]["sample_rate"])
    parts = Segmenter(assistant.cfg).split(clip)
    audio = np.concatenate(parts) if parts else clip
    n_turns = len(assistant.turns)
    text = assistant.handle_audio(audio, started_at=started)
    heard = assistant.heard[-1] if assistant.heard and assistant.heard[-1]["text"] == text else {"text": text, "action": "nothing heard"}
    turn = None
    if len(assistant.turns) > n_turns:                       # it was a command: wait for the reply
        turn = assistant.turns[-1]
        while turn["state"] not in ("done", "error"):
            time.sleep(0.05)
        assistant.speaker.wait()
    return {"heard": text, "action": heard["action"], "utterances": len(parts),
            "reply": turn["reply"] if turn else None, "tools": [e["tool"] for e in turn["tools"]] if turn else []}

Overwriting va/colab.py


In [14]:
# 5) Microphone access: record 2 seconds and print the volume. All zeros means the mic is blocked.
if IN_COLAB:   # the browser's microphone: allow access when the browser asks
    colab = fresh("va.colab")
    audio = colab.record_clip(2.0)
else:
    import sounddevice as sd
    print(sd.query_devices())
    audio = sd.rec(int(2 * 16000), samplerate=16000, channels=1, dtype="float32"); sd.wait()
print(f"mic RMS: {float(np.sqrt(np.mean(audio**2))):.5f}  (0.00000 = no access)")

<IPython.core.display.Javascript object>

mic RMS: 0.03652  (0.00000 = no access)


---
## Stage 1: A typed agent that manages appointments (30 minutes)

### The appointment store

SQLite, one table. Dates are stored as `YYYY-MM-DD HH:MM` strings, which sort correctly as text. The store validates the date and refuses an empty location, so a bad tool call comes back as an error the model can read instead of bad data.

In [15]:
%%writefile va/appointments.py
"""Appointments in SQLite. The agent's four tools call these methods."""
from __future__ import annotations

import sqlite3
import threading
from datetime import datetime
from pathlib import Path

DATE_FMT = "%Y-%m-%d %H:%M"
FIELDS = ("date", "location", "title", "description")


def normalize_date(value: str) -> str:
    """Accept 'YYYY-MM-DD HH:MM' (also with a T or seconds) and return 'YYYY-MM-DD HH:MM'."""
    text = str(value).strip().replace("T", " ")
    for fmt in (DATE_FMT, "%Y-%m-%d %H:%M:%S"):
        try:
            return datetime.strptime(text, fmt).strftime(DATE_FMT)
        except ValueError:
            pass
    raise ValueError(f"date must look like YYYY-MM-DD HH:MM, got {value!r}")


class AppointmentStore:
    def __init__(self, db_path: str | Path):
        self.db_path = str(db_path)
        if self.db_path != ":memory:":
            Path(self.db_path).parent.mkdir(parents=True, exist_ok=True)
        # One connection shared by the agent thread and the web page, guarded by a lock.
        self._conn = sqlite3.connect(self.db_path, check_same_thread=False)
        self._conn.row_factory = sqlite3.Row
        self._lock = threading.Lock()
        with self._lock, self._conn:
            self._conn.execute(
                """CREATE TABLE IF NOT EXISTS appointments (
                       id INTEGER PRIMARY KEY AUTOINCREMENT,
                       date TEXT NOT NULL,
                       location TEXT NOT NULL,
                       title TEXT NOT NULL,
                       description TEXT DEFAULT '',
                       created_at TEXT DEFAULT CURRENT_TIMESTAMP)"""
            )

    def create(self, date: str, location: str, title: str, description: str = "") -> dict:
        if not str(location or "").strip():
            raise ValueError("location is required")
        if not str(title or "").strip():
            raise ValueError("title is required")
        date = normalize_date(date)
        with self._lock, self._conn:
            cur = self._conn.execute(
                "INSERT INTO appointments (date, location, title, description) VALUES (?, ?, ?, ?)",
                (date, location.strip(), title.strip(), (description or "").strip()),
            )
            new_id = cur.lastrowid
        return self.get(new_id)

    def get(self, appointment_id: int) -> dict | None:
        with self._lock:
            row = self._conn.execute(
                "SELECT id, date, location, title, description FROM appointments WHERE id = ?",
                (int(appointment_id),),
            ).fetchone()
        return dict(row) if row else None

    def list(self) -> list[dict]:
        with self._lock:
            rows = self._conn.execute(
                "SELECT id, date, location, title, description FROM appointments ORDER BY date, id"
            ).fetchall()
        return [dict(r) for r in rows]

    def update(self, appointment_id: int, **changes) -> dict:
        changes = {k: v for k, v in changes.items() if k in FIELDS and v is not None}
        if not changes:
            raise ValueError("no fields to change")
        if "date" in changes:
            changes["date"] = normalize_date(changes["date"])
        if self.get(appointment_id) is None:
            raise KeyError(f"no appointment with id {appointment_id}")
        sets = ", ".join(f"{k} = ?" for k in changes)
        with self._lock, self._conn:
            self._conn.execute(f"UPDATE appointments SET {sets} WHERE id = ?",
                               (*changes.values(), int(appointment_id)))
        return self.get(appointment_id)

    def delete(self, appointment_id: int) -> dict:
        appt = self.get(appointment_id)
        if appt is None:
            raise KeyError(f"no appointment with id {appointment_id}")
        with self._lock, self._conn:
            self._conn.execute("DELETE FROM appointments WHERE id = ?", (int(appointment_id),))
        return appt

    def clear(self) -> None:
        with self._lock, self._conn:
            self._conn.execute("DELETE FROM appointments")

Overwriting va/appointments.py


### The agent

**Tools.** The four appointment tools have the fixed names and arguments from the exercise, plus `remember` (used in Stage 4). They are plain JSON schemas, so any Ollama tool model can use them.

**System prompt.** Rebuilt for every request with the current date and time and a list of the next seven dates (`- Friday: Friday 2026-11-13`). Small models are bad at calendar arithmetic; looking a weekday up in a list is easy. The rules from the exercise are numbered: ask for a missing location, never claim success without `"ok": true`, say the id, confirm deletes, one or two plain sentences.

**The loop.** Stream a reply; if the model asked for tools, run them, add the results as `tool` messages and ask again; stop when it answers in text. The text streams out through `on_text` (Stage 2 speaks it). Two details for small models:
- some write the tool call as JSON *text* instead of a real tool call. The `_TextGate` holds back text that starts with `{`, so it is never spoken, and `_parse_text_tool_call` runs it;
- the conversation history (including tool results) is kept, so "yes" and "move it to 4 PM" have context.

**Guards.** Prompts are requests; code is a guarantee. With `llm.guards: true` the tools themselves enforce the two rules small models break most:
- `create_appointment` refuses a location that does not appear in what the user said (or in memory), and tells the model to ask;
- `delete_appointment` refuses unless the user's latest message is a yes to a question from the assistant.

An identical booking in the same conversation returns the existing id rather than a duplicate. Set `guards: false` to see how the bare model does.

In [16]:
%%writefile va/agent.py
"""The agent: an Ollama chat model in a tool-calling loop, with appointments and memory.

Works with any Ollama model that supports tools; the model name comes from the config.
"""
from __future__ import annotations

import json
import logging
import re
import time
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Callable
from zoneinfo import ZoneInfo

import ollama

from .appointments import AppointmentStore

log = logging.getLogger("va.agent")

# ---------------------------------------------------------------------------
# Tool schemas. Names and arguments are fixed by the exercise.
# ---------------------------------------------------------------------------
_DATE = {"type": "string", "description": "Date and time as YYYY-MM-DD HH:MM, 24-hour clock"}
TOOLS = [
    {"type": "function", "function": {
        "name": "create_appointment",
        "description": "Save one new appointment. Only call this when the user said the date, time AND location.",
        "parameters": {"type": "object", "required": ["date", "location", "title"], "properties": {
            "date": _DATE,
            "location": {"type": "string", "description": "Where it is, exactly as the user said it"},
            "title": {"type": "string", "description": "Short title, e.g. 'Dentist'"},
            "description": {"type": "string", "description": "Optional extra details"}}}}},
    {"type": "function", "function": {
        "name": "list_appointments",
        "description": "Return all appointments ordered by date, with their ids.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "update_appointment",
        "description": "Change an existing appointment. Pass only the fields that change.",
        "parameters": {"type": "object", "required": ["appointment_id"], "properties": {
            "appointment_id": {"type": "integer", "description": "Id from list_appointments"},
            "date": _DATE,
            "location": {"type": "string"},
            "title": {"type": "string"},
            "description": {"type": "string"}}}}},
    {"type": "function", "function": {
        "name": "delete_appointment",
        "description": "Delete one appointment. Only call after the user answered yes to your confirmation question.",
        "parameters": {"type": "object", "required": ["appointment_id"], "properties": {
            "appointment_id": {"type": "integer", "description": "Id from list_appointments"}}}}},
    {"type": "function", "function": {
        "name": "remember",
        "description": "Store a lasting personal fact or preference about the user, e.g. 'The user prefers morning meetings'. Never for appointments.",
        "parameters": {"type": "object", "required": ["fact"], "properties": {
            "fact": {"type": "string", "description": "One short sentence about the user"}}}}},
]
TOOL_NAMES = {t["function"]["name"] for t in TOOLS}

SYSTEM_PROMPT = """You are {name}, a voice assistant that manages the user's appointments. Everything you write is read aloud.

Current date and time: {now}.
Dates for the next week:
{calendar}

Rules:
1. Tool dates are YYYY-MM-DD HH:MM on a 24-hour clock: 3 pm is 15:00, noon is 12:00. A weekday name means the next date with that name in the list above.
2. To book you need a date, a time, a title and a location. If the user did not say where, ask one short question about the location and do not call create_appointment. Never invent or guess a location.
3. Never say an appointment was booked, moved or deleted unless the tool you just called returned "ok": true. When you book, say the id from the tool result.
4. To move or change an appointment, call list_appointments to find its id, then call update_appointment with only the fields that change. Never create a new appointment to move one.
5. To delete: call list_appointments, then ask "Delete the <title> on <day> at <time>?" and stop. Call delete_appointment only after the user says yes. If they say no, keep it.
6. For questions about the calendar, call list_appointments and answer from its result.
7. When the user asks you to remember something about themselves, call remember. Appointments never go into memory.
8. Answer in one or two short sentences of plain text. No lists, no markdown, no emojis. Say times the way people speak, like 3 PM.
{memories}"""

_AFFIRM = re.compile(r"^\W*(yes|yeah|yep|yup|sure|ok|okay|confirm(ed)?|correct|go ahead|do it|please do|delete it|that's right|right)\b", re.I)
_THINK = re.compile(r"<think>.*?</think>", re.S)


@dataclass
class TurnResult:
    user: str
    reply: str = ""
    tool_events: list[dict] = field(default_factory=list)
    memories: list[dict] = field(default_factory=list)
    seconds: float = 0.0
    first_text_seconds: float | None = None
    model: str = ""


class _TextGate:
    """Streams text deltas out, unless the reply starts like a JSON tool call or a <think> block."""

    def __init__(self, sink: Callable[[str], None] | None):
        self.sink, self.mode, self.buf = sink, None, ""

    def feed(self, delta: str) -> None:
        if not delta:
            return
        self.buf += delta
        if self.mode is None:
            stripped = self.buf.lstrip()
            if not stripped:
                return
            self.mode = "hold" if stripped[0] in "{[`<" else "stream"
            if self.mode == "stream" and self.sink:
                self.sink(self.buf)
            return
        if self.mode == "stream" and self.sink:
            self.sink(delta)


def _parse_text_tool_call(content: str) -> list[dict]:
    """Small models sometimes write the tool call as JSON text instead of a real tool call."""
    text = content.strip().strip("`")
    text = re.sub(r"^json\s*", "", text)
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        return []
    objs = obj if isinstance(obj, list) else [obj]
    calls = []
    for o in objs:
        if isinstance(o, dict) and o.get("name") in TOOL_NAMES:
            args = o.get("arguments", o.get("parameters", {})) or {}
            calls.append({"name": o["name"], "arguments": args})
    return calls


class Agent:
    def __init__(self, cfg: dict, store: AppointmentStore, memory=None, model: str | None = None,
                 client=None, clock: Callable[[], datetime] | None = None):
        self.cfg, self.store, self.memory = cfg, store, memory
        self.model = model or cfg["llm"]["model"]
        self.client = client or ollama.Client(host=cfg["llm"]["host"])
        tz = cfg["assistant"].get("timezone")
        self.tz = ZoneInfo(tz) if tz else None
        self.clock = clock or (lambda: datetime.now(self.tz).replace(tzinfo=None))
        self.history: list[dict] = []
        self.guards = bool(cfg["llm"].get("guards", True))
        self._send_think = cfg["llm"].get("think") is not None

    # -- prompt ---------------------------------------------------------------
    def system_prompt(self, memories: list[dict]) -> str:
        now = self.clock()
        days = []
        for i in range(8):
            d = now + timedelta(days=i)
            label = "today" if i == 0 else "tomorrow" if i == 1 else d.strftime("%A")
            days.append(f"- {label}: {d:%A %Y-%m-%d}")
        mem = ""
        if memories:
            mem = "\nWhat you know about the user (from memory):\n" + "\n".join(f"- {m['text']}" for m in memories)
        return SYSTEM_PROMPT.format(name=self.cfg["assistant"]["name"], now=f"{now:%A, %Y-%m-%d %H:%M}",
                                    calendar="\n".join(days), memories=mem)

    def warm_up(self) -> None:
        """Load the model into memory now, so the first command is fast."""
        self.client.chat(model=self.model, messages=[{"role": "user", "content": "hi"}],
                         options={"num_predict": 1}, keep_alive=self.cfg["llm"].get("keep_alive"))

    def reset(self) -> None:
        self.history.clear()

    # -- tools -----------------------------------------------------------------
    def _recent_user_text(self, current: str, memories: list[dict]) -> str:
        users = [m["content"] for m in self.history if m["role"] == "user"][-3:]
        return " ".join(users + [current] + [m["text"] for m in memories]).lower()

    def _last_assistant_text(self) -> str:
        for m in reversed(self.history):
            if m["role"] == "assistant" and m.get("content"):
                return m["content"]
        return ""

    def call_tool(self, name: str, args: dict, ctx: dict) -> dict:
        """Run one tool and return a JSON-able result with "ok"."""
        try:
            if name == "create_appointment":
                location = str(args.get("location") or "").strip()
                if not location:
                    return {"ok": False, "error": "No location given. Ask the user where it is."}
                if self.guards and not _grounded(location, ctx["user_text"]):
                    return {"ok": False, "error": f"The user never said the location {location!r}. Ask the user where it is."}
                if self.guards:   # the same booking twice in one conversation -> return the first one
                    for a in self.store.list():
                        if (a["date"], a["title"].lower(), a["location"].lower()) == (
                                _safe_date(args.get("date")), str(args.get("title", "")).strip().lower(), location.lower()):
                            return {"ok": True, "id": a["id"], "appointment": a, "note": "already booked"}
                appt = self.store.create(args.get("date", ""), location, args.get("title", ""), args.get("description") or "")
                return {"ok": True, "id": appt["id"], "appointment": appt}
            if name == "list_appointments":
                appts = self.store.list()
                return {"ok": True, "count": len(appts), "appointments": appts}
            if name == "update_appointment":
                aid = int(args.get("appointment_id"))
                changes = {k: args[k] for k in ("date", "location", "title", "description") if args.get(k)}
                return {"ok": True, "appointment": self.store.update(aid, **changes)}
            if name == "delete_appointment":
                aid = int(args.get("appointment_id"))
                appt = self.store.get(aid)
                if appt is None:
                    return {"ok": False, "error": f"No appointment with id {aid}."}
                if self.guards and not (_AFFIRM.search(ctx["user_text_now"]) and "?" in self._last_assistant_text()):
                    return {"ok": False, "error": "Not deleted yet. First ask the user to confirm, e.g. "
                            f"'Delete the {appt['title']} on {appt['date']}?', and wait for yes.", "appointment": appt}
                return {"ok": True, "deleted": self.store.delete(aid)}
            if name == "remember":
                if self.memory is None:
                    return {"ok": False, "error": "Memory is not available."}
                res = self.memory.add(str(args.get("fact", "")), source="user")
                return {"ok": res.get("stored", False) or res.get("reason") == "already known", **res}
            return {"ok": False, "error": f"Unknown tool {name}."}
        except (ValueError, KeyError, TypeError) as exc:
            return {"ok": False, "error": str(exc)}

    # -- the loop ----------------------------------------------------------------
    def run(self, user_text: str, on_text: Callable[[str], None] | None = None,
            on_tool: Callable[[dict], None] | None = None) -> TurnResult:
        """One user turn. `on_text` gets reply text as it streams; `on_tool` gets each tool call and result."""
        t0 = time.perf_counter()
        result = TurnResult(user=user_text, model=self.model)
        memories = []
        if self.memory is not None:
            try:
                memories = self.memory.search(user_text)
            except Exception:
                log.exception("memory search failed")
        result.memories = memories
        ctx = {"user_text": self._recent_user_text(user_text, memories), "user_text_now": user_text}

        def text_sink(delta: str) -> None:
            if result.first_text_seconds is None:
                result.first_text_seconds = time.perf_counter() - t0
            if on_text:
                on_text(delta)

        turn = [{"role": "user", "content": user_text}]
        kwargs = {"options": {"temperature": self.cfg["llm"].get("temperature", 0)},
                  "keep_alive": self.cfg["llm"].get("keep_alive")}
        reply = ""
        for _ in range(int(self.cfg["llm"].get("max_tool_rounds", 6))):
            messages = [{"role": "system", "content": self.system_prompt(memories)}] + self.history + turn
            gate, content, calls = _TextGate(text_sink), "", []
            for chunk in self._stream(messages, kwargs):
                msg = chunk.message
                if msg.content:
                    content += msg.content
                    gate.feed(msg.content)
                for tc in msg.tool_calls or []:
                    args = tc.function.arguments
                    if isinstance(args, str):
                        args = json.loads(args or "{}")
                    calls.append({"name": tc.function.name, "arguments": dict(args or {})})
            content = _THINK.sub("", content).strip()
            if not calls and gate.mode == "hold":
                calls = _parse_text_tool_call(content)
                if calls:
                    content = ""
                elif on_text and content:   # held text that was not a tool call after all
                    text_sink(content)
            if not calls:
                reply = content
                break
            turn.append({"role": "assistant", "content": content,
                         "tool_calls": [{"function": {"name": c["name"], "arguments": c["arguments"]}} for c in calls]})
            for c in calls:
                out = self.call_tool(c["name"], c["arguments"], ctx)
                event = {"tool": c["name"], "arguments": c["arguments"], "result": out}
                result.tool_events.append(event)
                log.info("tool %s(%s) -> %s", c["name"], json.dumps(c["arguments"]), json.dumps(out)[:300])
                if on_tool:
                    on_tool(event)
                turn.append({"role": "tool", "tool_name": c["name"], "content": json.dumps(out)})
        else:
            reply = "Sorry, I could not finish that."
            text_sink(reply)

        if not reply:
            reply = "Sorry, I did not get that."
            text_sink(reply)
        turn.append({"role": "assistant", "content": reply})
        self.history.extend(turn)
        self._trim_history()
        result.reply = reply
        result.seconds = time.perf_counter() - t0
        log.info("turn model=%s %.2fs user=%r reply=%r", self.model, result.seconds, user_text, reply)
        return result

    def _stream(self, messages: list[dict], kwargs: dict):
        """Stream one model reply. Models that reject the `think` flag are retried without it."""
        if self._send_think:
            kwargs = {**kwargs, "think": self.cfg["llm"]["think"]}
        try:
            yield from self.client.chat(model=self.model, messages=messages, tools=TOOLS, stream=True, **kwargs)
        except ollama.ResponseError as exc:
            if "think" not in kwargs or "think" not in str(exc).lower():
                raise
            log.info("%s does not take the think flag; not sending it again", self.model)
            self._send_think = False
            kwargs = {k: v for k, v in kwargs.items() if k != "think"}
            yield from self.client.chat(model=self.model, messages=messages, tools=TOOLS, stream=True, **kwargs)

    def _trim_history(self) -> None:
        keep = int(self.cfg["assistant"].get("history_turns", 8))
        starts = [i for i, m in enumerate(self.history) if m["role"] == "user"]
        if len(starts) > keep:
            self.history = self.history[starts[-keep]:]


def _safe_date(value) -> str:
    from .appointments import normalize_date
    try:
        return normalize_date(value)
    except ValueError:
        return ""


_STOP = {"the", "a", "an", "at", "in", "on", "of", "to", "my", "street", "st", "ave", "avenue", "road", "rd"}


def _grounded(location: str, said: str) -> bool:
    """True if some word of the location was actually said by the user (or is in memory)."""
    words = [w for w in re.findall(r"[a-z0-9]+", location.lower()) if w not in _STOP]
    if not words:
        return True
    said_words = set(re.findall(r"[a-z0-9]+", said))
    return any(w in said_words for w in words)

Overwriting va/agent.py


### The checks

`checks.py` types the eight commands into a fresh agent with an empty in-memory database and **checks the stored data, not the reply**, after each one. It is reused by Stage 6.

In [17]:
%%writefile checks.py
"""Runnable checks for Stages 1, 2, 4 and the Stage 6 measurement.

    python checks.py stage1                 # the eight typed checks with the model in config.yaml
    python checks.py stage1 --model qwen3:8b
    python checks.py stage2                 # speech latency and stop
    python checks.py stage4                 # memory survives a restart
    python checks.py stage6                 # stage 1 on every model in llm.compare_models, writes the README table
"""
from __future__ import annotations

import argparse
import re
import sys
import time
from datetime import datetime, timedelta

from va.agent import Agent
from va.appointments import AppointmentStore
from va.config import load_config, setup_logging, with_overrides


def _next_weekday(now: datetime, weekday: int) -> set[str]:
    """Dates that count as 'on Friday': the next Friday (and the one after if today is Friday)."""
    days = (weekday - now.weekday()) % 7
    out = {(now + timedelta(days=days or 7)).strftime("%Y-%m-%d")}
    if days == 0:
        out.add(now.strftime("%Y-%m-%d"))
    return out


def stage1_cases(now: datetime):
    tomorrow = (now + timedelta(days=1)).strftime("%Y-%m-%d")
    fridays = _next_weekday(now, 4)

    def dentist(rows):
        return [r for r in rows if "dentist" in (r["title"] + " " + r["description"]).lower()]

    def c1(before, after, reply, ctx):
        d = dentist(after)
        ok = len(after) == 1 and len(d) == 1 and d[0]["date"] == f"{tomorrow} 15:00" and "chicago" in d[0]["location"].lower()
        if d:
            ctx["dentist_id"] = d[0]["id"]
        return ok, f"rows={after}"

    def c2(before, after, reply, ctx):
        aid = ctx.get("dentist_id")
        return aid is not None and re.search(rf"\b{aid}\b", ctx["reply1"]) is not None, f"id={aid} reply={ctx['reply1']!r}"

    def c3(before, after, reply, ctx):
        return len(after) == len(before) and "?" in reply, f"new rows={len(after) - len(before)}"

    def c4(before, after, reply, ctx):
        return "dentist" in reply.lower(), ""

    def c5(before, after, reply, ctx):
        d = dentist(after)
        ok = len(after) == len(before) and len(d) == 1 and d[0]["id"] == ctx.get("dentist_id") and d[0]["date"] == f"{tomorrow} 16:00"
        return ok, f"rows={after}"

    def c6(before, after, reply, ctx):
        return len(dentist(after)) == 1 and "?" in reply, f"still there={len(dentist(after)) == 1}"

    def c7(before, after, reply, ctx):
        return len(dentist(after)) == 0, f"rows={after}"

    def c8(before, after, reply, ctx):
        new = [r for r in after if r["id"] not in {b["id"] for b in before}]
        ok = (len(new) == 1 and new[0]["date"][:10] in fridays and new[0]["date"].endswith("12:00")
              and "roma" in new[0]["location"].lower())
        return ok, f"new={new}"

    return [
        ("book a dentist appointment tomorrow at 3 pm in Chicago", "one appointment, tomorrow 15:00, Chicago", c1),
        (None, "the id is in the answer", c2),
        ("book a haircut on Friday at 10 am", "nothing saved, the agent asks where", c3),
        ("what is on my calendar", "the dentist appointment is mentioned", c4),
        ("move my dentist appointment to 4 pm", "same appointment, now 16:00, no duplicate", c5),
        ("delete my dentist appointment", "asks first, still there", c6),
        ("yes", "deleted", c7),
        ("book lunch with Sam on Friday at noon at Cafe Roma", "exactly one appointment", c8),
    ]


def run_stage1(cfg: dict, model: str | None = None, verbose: bool = True) -> dict:
    """Type the eight commands into a fresh agent with an empty database and check the stored data."""
    store = AppointmentStore(":memory:")
    agent = Agent(cfg, store, memory=None, model=model)
    agent.warm_up()
    now = datetime.now()
    ctx, rows, passed, total = {}, [], [], 0.0
    for i, (text, expected, check) in enumerate(stage1_cases(now), 1):
        before = store.list()
        reply = ctx.get("reply1", "")
        if text is not None:
            res = agent.run(text)
            total += res.seconds
            reply = res.reply
            if i == 1:
                ctx["reply1"] = reply
            tools = ", ".join(e["tool"] + ("" if e["result"].get("ok") else "!") for e in res.tool_events) or "-"
        else:
            tools = "-"
        ok, detail = check(before, store.list(), reply, ctx)
        passed.append(ok)
        rows.append({"#": i, "typed": text or "(reply)", "expected": expected, "pass": ok,
                     "tools": tools, "reply": reply, "detail": detail})
        if verbose:
            print(f"{'PASS' if ok else 'FAIL'} {i}. {text or '(reply)'}\n     tools: {tools}\n     reply: {reply}")
            if not ok:
                print(f"     {detail}")
    summary = {"model": agent.model, "passed": sum(passed), "seconds": round(total, 1), "rows": rows}
    if verbose:
        print(f"\n{agent.model}: {summary['passed']}/8 passed in {summary['seconds']} s")
    return summary


def run_stage2(cfg: dict) -> dict:
    """Time to first speech for a booking, and how fast stop() silences a sentence.

    Uses only Stage 1 and 2 code (agent + speaker), with an empty in-memory database."""
    from va.speech_out import Speaker
    agent = Agent(cfg, AppointmentStore(":memory:"), memory=None)
    agent.warm_up()
    speaker = Speaker(cfg)
    speaker.reset_timer()
    t0 = time.perf_counter()
    reply = agent.run("book a dentist appointment tomorrow at 3 pm in Chicago", on_text=speaker.feed).reply
    speaker.flush()
    while speaker.first_audio_at is None and time.perf_counter() - t0 < 60:
        time.sleep(0.01)
    if speaker.first_audio_at is None:
        raise TimeoutError("no speech after 60 s")
    first = speaker.first_audio_at - t0
    print("reply:", reply)
    speaker.wait()
    speaker.say("This is a long sentence that keeps going and going so that we have time to interrupt it in the middle.")
    time.sleep(1.0)
    t1 = time.perf_counter()
    speaker.stop()
    while not speaker.silent():
        time.sleep(0.005)
    stop = time.perf_counter() - t1
    print(f"first words after {first:.2f} s (need < 4)   ->  {'PASS' if first < 4 else 'FAIL'}")
    print(f"silent {stop * 1000:.0f} ms after stop()      ->  {'PASS' if stop < 0.5 else 'FAIL'}")
    return {"first_speech_s": round(first, 2), "stop_s": round(stop, 3)}


def run_stage4(cfg: dict) -> bool:
    """Remember a preference, 'restart' (new objects, nothing shared but Qdrant), then ask about it."""
    from va.memory import MemoryStore
    cfg = with_overrides(cfg, **{"memory.collection": cfg["memory"]["collection"] + "_check"})
    MemoryStore(cfg).clear()

    def fresh_agent():
        mem = MemoryStore(cfg)
        return Agent(cfg, AppointmentStore(":memory:"), memory=mem), mem

    agent, mem = fresh_agent()
    r1 = agent.run("remember that I prefer morning meetings")
    print("reply 1:", r1.reply, "| tools:", [e["tool"] for e in r1.tool_events])
    print("memories:", [m["text"] for m in mem.all()])
    del agent, mem                               # the "restart": no history, no objects survive
    agent, mem = fresh_agent()
    r2 = agent.run("what time of day do I prefer for meetings")
    ok = "morning" in r2.reply.lower()
    print("reply 2:", r2.reply, "| memory used:", [m["text"] for m in r2.memories])
    print("PASS" if ok else "FAIL")
    return ok


def write_readme_table(results: list[dict], path="README.md") -> str:
    lines = ["| Model | Checks passed (of 8) | Total seconds for the eight checks |", "|---|---|---|"]
    lines += [f"| {r['model']} | {r['passed']} | {r['seconds']} |" for r in results]
    table = "\n".join(lines)
    text = open(path).read()
    new = re.sub(r"(<!-- stage6 -->\n).*?(\n<!-- /stage6 -->)", lambda m: m.group(1) + table + m.group(2), text, flags=re.S)
    open(path, "w").write(new)
    return table


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("stage", choices=["stage1", "stage2", "stage4", "stage6"])
    p.add_argument("--model")
    p.add_argument("--config")
    args = p.parse_args()
    cfg = load_config(args.config)
    setup_logging(cfg)
    if args.stage == "stage1":
        sys.exit(0 if run_stage1(cfg, args.model)["passed"] == 8 else 1)
    if args.stage == "stage2":
        run_stage2(cfg)
    if args.stage == "stage4":
        run_stage4(cfg)
    if args.stage == "stage6":
        results = [run_stage1(cfg, m) for m in cfg["llm"]["compare_models"]]
        print(write_readme_table(results))

Overwriting checks.py


### Check 1: all eight must pass

In [18]:
config, checks = fresh("va.config", "checks")
cfg = config.load_config()
config.setup_logging(cfg)
stage1 = checks.run_stage1(cfg)

INFO:va.agent:tool create_appointment({"title": "Dentist", "date": "2026-11-13 15:00", "location": "Chicago"}) -> {"ok": true, "id": 1, "appointment": {"id": 1, "date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist", "description": ""}}
INFO:va.agent:turn model=qwen3:8b 4.14s user='book a dentist appointment tomorrow at 3 pm in Chicago' reply='Your dentist appointment is booked for tomorrow at 3 PM in Chicago with ID 1.'


PASS 1. book a dentist appointment tomorrow at 3 pm in Chicago
     tools: create_appointment
     reply: Your dentist appointment is booked for tomorrow at 3 PM in Chicago with ID 1.
PASS 2. (reply)
     tools: -
     reply: Your dentist appointment is booked for tomorrow at 3 PM in Chicago with ID 1.


INFO:va.agent:tool create_appointment({"date": "2026-11-13 10:00", "location": "/no_think", "title": "Haircut"}) -> {"ok": false, "error": "The user never said the location '/no_think'. Ask the user where it is."}
INFO:va.agent:turn model=qwen3:8b 2.29s user='book a haircut on Friday at 10 am' reply='Where is your haircut appointment?'


PASS 3. book a haircut on Friday at 10 am
     tools: create_appointment!
     reply: Where is your haircut appointment?


INFO:va.agent:tool list_appointments({}) -> {"ok": true, "count": 1, "appointments": [{"id": 1, "date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist", "description": ""}]}
INFO:va.agent:turn model=qwen3:8b 1.44s user='what is on my calendar' reply='You have a dentist appointment on Friday at 3 PM in Chicago.'


PASS 4. what is on my calendar
     tools: list_appointments
     reply: You have a dentist appointment on Friday at 3 PM in Chicago.


... (4 outputs left out)

INFO:va.agent:tool delete_appointment({"appointment_id": 1}) -> {"ok": true, "deleted": {"id": 1, "date": "2026-11-13 16:00", "location": "Chicago", "title": "Dentist", "description": ""}}
INFO:va.agent:turn model=qwen3:8b 1.76s user='yes' reply='Your dentist appointment on Friday at 4 PM in Chicago has been deleted.'


PASS 7. yes
     tools: delete_appointment
     reply: Your dentist appointment on Friday at 4 PM in Chicago has been deleted.


INFO:va.agent:tool create_appointment({"location": "Cafe Roma", "title": "Lunch with Sam", "date": "2026-11-13 12:00"}) -> {"ok": true, "id": 2, "appointment": {"id": 2, "date": "2026-11-13 12:00", "location": "Cafe Roma", "title": "Lunch with Sam", "description": ""}}
INFO:va.agent:turn model=qwen3:8b 5.00s user='book lunch with Sam on Friday at noon at Cafe Roma' reply='Your lunch with Sam is booked for Friday at noon at Cafe Roma with ID 2.'


PASS 8. book lunch with Sam on Friday at noon at Cafe Roma
     tools: create_appointment
     reply: Your lunch with Sam is booked for Friday at noon at Cafe Roma with ID 2.

qwen3:8b: 8/8 passed in 20.0 s


In [19]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(stage1["rows"])[["#", "typed", "pass", "tools", "reply"]]

,#,typed,pass,tools,reply
0,1,book a dentist appointment tomorrow at 3 pm in Chicago,True,create_appointment,Your dentist appointment is booked for tomorrow at 3 PM in Chicago with ID 1.
1,2,(reply),True,-,Your dentist appointment is booked for tomorrow at 3 PM in Chicago with ID 1.
2,3,book a haircut on Friday at 10 am,True,create_appointment!,Where is your haircut appointment?
3,4,what is on my calendar,True,list_appointments,You have a dentist appointment on Friday at 3 PM in Chicago.
4,5,move my dentist appointment to 4 pm,True,"list_appointments, update_appointment",Your dentist appointment is now at 4 PM on Friday in Chicago with ID 1.
5,6,delete my dentist appointment,True,list_appointments,Delete the dentist appointment on Friday at 4 PM?
6,7,yes,True,delete_appointment,Your dentist appointment on Friday at 4 PM in Chicago has been deleted.
7,8,book lunch with Sam on Friday at noon at Cafe Roma,True,create_appointment,Your lunch with Sam is booked for Friday at noon at Cafe Roma with ID 2.


Try your own commands. A tool name ending in `!` in the table above means the tool refused (a guard or a validation error).

In [20]:
config, agent_mod, appts = fresh("va.config", "va.agent", "va.appointments")
store = appts.AppointmentStore(":memory:")
agent = agent_mod.Agent(cfg, store)
for text in ["book a dentist appointment tomorrow at 3 pm in Chicago", "what is on my calendar"]:
    r = agent.run(text)
    print(f"you> {text}\n  tools: {[(e['tool'], e['result'].get('ok')) for e in r.tool_events]}\nloop> {r.reply}  ({r.seconds:.1f} s)\n")
store.list()

INFO:va.agent:tool create_appointment({"title": "Dentist", "date": "2026-11-13 15:00", "location": "Chicago"}) -> {"ok": true, "id": 1, "appointment": {"id": 1, "date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist", "description": ""}}
INFO:va.agent:turn model=qwen3:8b 5.93s user='book a dentist appointment tomorrow at 3 pm in Chicago' reply='Your dentist appointment is booked for tomorrow at 3 PM in Chicago with ID 1.'


you> book a dentist appointment tomorrow at 3 pm in Chicago
  tools: [('create_appointment', True)]
loop> Your dentist appointment is booked for tomorrow at 3 PM in Chicago with ID 1.  (5.9 s)



INFO:va.agent:tool list_appointments({}) -> {"ok": true, "count": 1, "appointments": [{"id": 1, "date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist", "description": ""}]}
INFO:va.agent:turn model=qwen3:8b 1.61s user='what is on my calendar' reply='You have one appointment: Dentist on Friday at 3 PM in Chicago.'


you> what is on my calendar
  tools: [('list_appointments', True)]
loop> You have one appointment: Dentist on Friday at 3 PM in Chicago.  (1.6 s)



[{'id': 1,
  'date': '2026-11-13 15:00',
  'location': 'Chicago',
  'title': 'Dentist',
  'description': ''}]

---
## Stage 2: Speech out (15 minutes)

- `clean_for_speech` strips what a voice reads badly: asterisks, hashes, backticks, bullets, links, emojis.
- `SentenceSplitter` takes the streamed text and returns each sentence as soon as it ends (`.`, `!`, `?` followed by a space, or a newline). It does not split after `Dr.` or `St.`, or inside `3.5`.
- `Speaker` speaks sentences in order on its own thread. Each sentence is a child process (`say`, `espeak-ng`, or `pyttsx3` run with `python -c`), so `stop()` empties the queue and kills the process: silence is immediate on every platform.
- On Colab (`tts.backend: notebook`) each sentence becomes a WAV made by `espeak-ng` and autoplays in the cell output. The speaker counts as "speaking" for the length of the clip, and `stop()` replaces the player with "(speech stopped)".

In [21]:
%%writefile va/speech_out.py
"""Text to speech: split streamed text into sentences, speak each one as soon as it is complete.

Every backend runs as a child process, so stop() can kill it at once. The "notebook" backend
(for Colab, which has no speaker) makes a WAV with espeak-ng and autoplays it in the cell output;
stop() removes the player.
"""
from __future__ import annotations

import logging
import queue
import re
import shutil
import subprocess
import sys
import tempfile
import threading
import time
import wave
from typing import Callable

log = logging.getLogger("va.tts")

# ---------------------------------------------------------------------------
# Text clean-up and sentence splitting
# ---------------------------------------------------------------------------
_ABBREV = {"dr", "mr", "mrs", "ms", "st", "ave", "jr", "sr", "vs", "etc", "e.g", "i.e", "a.m", "p.m", "no"}
_BOUNDARY = re.compile(r"[.!?]+[\"')\]]*\s+|\n+")


def clean_for_speech(text: str) -> str:
    """Remove things a voice would read badly: markdown, code ticks, bullets, links, emojis."""
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)          # [label](url) -> label
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"[*`_~]+", "", text)
    text = re.sub(r"[#>|]+", " ", text)
    text = re.sub(r"^\s*[-•]\s+", "", text, flags=re.M)               # bullets
    text = re.sub(r"[\U0001F300-\U0001FAFF☀-➿]", "", text)  # emojis
    return re.sub(r"\s+", " ", text).strip()


class SentenceSplitter:
    """Feed streamed text with feed(); get back the sentences that are complete so far."""

    def __init__(self):
        self.buf = ""

    def feed(self, delta: str) -> list[str]:
        self.buf += delta
        out, start = [], 0
        for m in _BOUNDARY.finditer(self.buf):
            piece = self.buf[start:m.end()]
            last_word = re.findall(r"([A-Za-z.]+)[.!?]+[\"')\]]*\s*$", piece)
            if last_word and last_word[0].lower().rstrip(".") in _ABBREV and "\n" not in m.group():
                continue                                          # "Dr. Smith" is not the end
            sentence = clean_for_speech(piece)
            if sentence:
                out.append(sentence)
            start = m.end()
        self.buf = self.buf[start:]
        return out

    def flush(self) -> list[str]:
        rest, self.buf = clean_for_speech(self.buf), ""
        return [rest] if rest else []


# ---------------------------------------------------------------------------
# Speaker
# ---------------------------------------------------------------------------
_PYTTSX3_SCRIPT = (
    "import sys, pyttsx3; e = pyttsx3.init(); e.setProperty('rate', int(sys.argv[2])); "
    "e.say(sys.argv[1]); e.runAndWait()"
)


def pick_backend(name: str = "auto") -> str:
    if name != "auto":
        return name
    if sys.platform == "darwin":
        return "say"
    if shutil.which("espeak-ng") or shutil.which("espeak"):
        return "espeak"
    return "pyttsx3"


class Speaker:
    """Speaks sentences in order on a background thread. stop() silences it immediately."""

    def __init__(self, cfg: dict, on_start: Callable[[], None] | None = None,
                 on_done: Callable[[], None] | None = None, enabled: bool = True):
        t = cfg["tts"]
        self.backend, self.voice, self.rate = pick_backend(t.get("backend", "auto")), t.get("voice"), int(t.get("rate", 190))
        self.enabled = enabled
        self.on_start, self.on_done = on_start, on_done
        self._q: queue.Queue = queue.Queue()
        self._proc: subprocess.Popen | None = None
        self._handle = None            # notebook backend: the display handle of the playing audio
        self._lock = threading.Lock()
        self._generation = 0           # bumped by stop(); stale sentences are dropped
        self._busy = False
        self.last_done = 0.0           # time.monotonic() when speech last finished
        self.first_audio_at: float | None = None   # time.perf_counter() of the first sentence handed to the voice
        self._splitter = SentenceSplitter()
        threading.Thread(target=self._worker, name="speaker", daemon=True).start()

    # -- streaming interface --------------------------------------------------
    def feed(self, delta: str) -> None:
        """Give streamed reply text; complete sentences start speaking right away."""
        for s in self._splitter.feed(delta):
            self.say(s)

    def flush(self) -> None:
        for s in self._splitter.flush():
            self.say(s)

    def say(self, sentence: str) -> None:
        sentence = clean_for_speech(sentence)
        if sentence:
            with self._lock:
                self._busy = True
            self._q.put((self._generation, sentence))

    def reset_timer(self) -> None:
        self.first_audio_at = None

    @property
    def speaking(self) -> bool:
        return self._busy

    def wait(self, timeout: float = 60) -> None:
        """Block until everything queued has been spoken (or stopped)."""
        end = time.monotonic() + timeout
        while self.speaking and time.monotonic() < end:
            time.sleep(0.02)

    def stop(self) -> None:
        """Cut speech off now and drop everything queued."""
        with self._lock:
            self._generation += 1
            self._splitter = SentenceSplitter()
            proc = self._proc
        try:
            while True:
                self._q.get_nowait()
        except queue.Empty:
            pass
        if proc and proc.poll() is None:
            proc.kill()
        self._silence_notebook()
        self._set_idle()

    def silent(self) -> bool:
        """True when no sound is playing right now."""
        proc = self._proc
        return (proc is None or proc.poll() is not None) and self._handle is None

    # -- internals ------------------------------------------------------------
    def _command(self, text: str) -> list[str]:
        if self.backend == "say":
            cmd = ["say", "-r", str(self.rate)]
            return cmd + (["-v", self.voice] if self.voice else []) + [text]
        if self.backend == "espeak":
            exe = shutil.which("espeak-ng") or "espeak"
            return [exe, "-s", str(self.rate)] + (["-v", self.voice] if self.voice else []) + [text]
        if self.backend == "pyttsx3":
            return [sys.executable, "-c", _PYTTSX3_SCRIPT, text, str(self.rate)]
        raise ValueError(f"unknown tts backend {self.backend!r}")

    # -- notebook backend (Colab) ------------------------------------------------
    def _play_in_notebook(self, text: str, gen: int) -> None:
        try:
            from IPython import get_ipython
            from IPython.display import Audio, display
        except ImportError:
            get_ipython = lambda: None   # noqa: E731
        exe = shutil.which("espeak-ng") or shutil.which("espeak")
        if exe is None or get_ipython() is None:      # e.g. inside the Streamlit process
            if self.first_audio_at is None:
                self.first_audio_at = time.perf_counter()
            print("[speaks]", text)
            return
        path = tempfile.mktemp(suffix=".wav")
        with self._lock:
            self._proc = subprocess.Popen([exe, "-s", str(self.rate), "-w", path, text],
                                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        self._proc.wait()
        if gen != self._generation:
            return
        with wave.open(path) as w:
            seconds = w.getnframes() / w.getframerate()
        handle = display(Audio(path, autoplay=True), display_id=True)
        if self.first_audio_at is None:
            self.first_audio_at = time.perf_counter()
        with self._lock:
            self._handle = handle
        end = time.monotonic() + seconds               # "speaking" lasts as long as the clip
        while time.monotonic() < end and gen == self._generation:
            time.sleep(0.02)
        with self._lock:
            if self._handle is handle:
                self._handle = None

    def _silence_notebook(self) -> None:
        with self._lock:
            handle, self._handle = self._handle, None
        if handle is not None:
            from IPython.display import HTML
            handle.update(HTML("<i>(speech stopped)</i>"))   # removing the player stops the sound

    def _set_idle(self) -> None:
        with self._lock:
            was_busy, self._busy = self._busy, False
            self.last_done = time.monotonic()
        if was_busy and self.on_done:
            self.on_done()

    def _worker(self) -> None:
        while True:
            gen, text = self._q.get()
            if gen != self._generation:
                continue
            if self.on_start:
                self.on_start()
            log.debug("say: %s", text)
            try:   # a broken voice must never kill this thread
                if not self.enabled:
                    raise RuntimeError("speech is off")
                if self.backend == "notebook":
                    self._play_in_notebook(text, gen)
                else:
                    if self.first_audio_at is None:
                        self.first_audio_at = time.perf_counter()
                    with self._lock:
                        self._proc = subprocess.Popen(self._command(text), stdout=subprocess.DEVNULL,
                                                      stderr=subprocess.DEVNULL)
                    self._proc.wait()
            except Exception as exc:
                if self.enabled:
                    log.error("TTS backend %r failed (%s); printing instead", self.backend, exc)
                if self.first_audio_at is None:
                    self.first_audio_at = time.perf_counter()
                print("[speaks]", text)
            if self._q.empty() and gen == self._generation:
                self._set_idle()

Overwriting va/speech_out.py


In [22]:
speech_out = fresh("va.speech_out")
splitter = speech_out.SentenceSplitter()
for delta in ["Booked your **dentist** ", "appointment with Dr. Lee at ", "3 PM. The id ", "is 4! Anything ", "else?"]:
    print(f"{delta!r:40} -> {splitter.feed(delta)}")
print(f"{'(end)':40} -> {splitter.flush()}")
print(speech_out.clean_for_speech("## Done! *Moved* `id 4` to [4 PM](http://x) 🎉"))

'Booked your **dentist** '               -> []
'appointment with Dr. Lee at '           -> []
'3 PM. The id '                          -> ['Booked your dentist appointment with Dr. Lee at 3 PM.']
'is 4! Anything '                        -> ['The id is 4!']
'else?'                                  -> []
(end)                                    -> ['Anything else?']
Done! Moved id 4 to 4 PM


### Check 2: first words in under 4 seconds, silent within half a second of `stop()`

Uses a separate database (`data/check_stage2.db`). Turn your speakers on. On Colab the replies play in this cell's output.

In [23]:
config, checks = fresh("va.config", "checks")
stage2 = checks.run_stage2(config.load_config())

INFO:va.agent:tool create_appointment({"date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist"}) -> {"ok": true, "id": 1, "appointment": {"id": 1, "date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist", "description": ""}}
INFO:va.agent:turn model=qwen3:8b 2.82s user='book a dentist appointment tomorrow at 3 pm in Chicago' reply='Your dentist appointment is booked for tomorrow at 3 PM in Chicago with ID 1.'


reply: Your dentist appointment is booked for tomorrow at 3 PM in Chicago with ID 1.


first words after 2.85 s (need < 4)   ->  PASS
silent 1 ms after stop()      ->  PASS


---
## Stage 3: Speech in and the wake word (30 minutes)

### Wake word

Speech-to-text output is lower-cased, split into words and punctuation, and the wake word is matched against a list of spellings (`wake.variants` in the config: `loop`, `lupe`, `loup`, `lube`, ...). It only counts:
- at the **start of a sentence**, possibly after fillers (`okay`, `hey`, `so`): "Okay, loop, book ...", "... tomorrow at 3. Loop. Book ...";
- or as a **vocative** between commas: "..., loop, book ...".

In "I like to loop videos" it is in the middle of a clause, so it is ignored. The command is whatever follows the wake word. Just "Loop." makes it answer "Yes?" and open the follow-up window.

In [24]:
%%writefile va/wake.py
"""Wake word matching on speech-to-text output.

Accepts:  "Loop, book ...", "Okay, loop, book ...", "... tomorrow at 3. Loop. Book ..."
Ignores:  "I like to loop videos"   (the wake word is in the middle of a clause)
"""
from __future__ import annotations

import re

_TOKEN = re.compile(r"[A-Za-z0-9']+|[.!?;:,]")
_SENTENCE_END = set(".!?;:")


class WakeWord:
    def __init__(self, cfg: dict):
        w = cfg["wake"]
        self.variants = {v.lower() for v in w["variants"]} | {w["word"].lower()}
        self.fillers = {f.lower() for f in w.get("fillers", [])}

    def find(self, text: str) -> str | None:
        """Return the command after the wake word ("" if only the wake word), or None if there is no wake word."""
        spans = [(m.group(), m.start(), m.end()) for m in _TOKEN.finditer(text)]
        for i, (tok, _, end) in enumerate(spans):
            if tok.lower() not in self.variants:
                continue
            # Words since the start of this sentence, before the wake word.
            j, before = i - 1, []
            while j >= 0 and spans[j][0] not in _SENTENCE_END:
                if spans[j][0] != ",":
                    before.append(spans[j][0].lower())
                j -= 1
            followed_by_punct = i + 1 < len(spans) and spans[i + 1][0] in _SENTENCE_END | {","}
            prev_is_comma = i > 0 and spans[i - 1][0] == ","
            at_start = all(b in self.fillers for b in before)        # "Loop ...", "Okay, loop ..."
            vocative = prev_is_comma and followed_by_punct          # "..., loop, book ..."
            if at_start or vocative:
                return text[end:].lstrip(" ,.!?;:-").strip()
        return None

Writing va/wake.py


In [25]:
config, wake = fresh("va.config", "va.wake")
w = wake.WakeWord(config.load_config())
cases = {
    "Loop, book a dentist appointment tomorrow at 3 PM in Chicago.": "book a dentist appointment tomorrow at 3 PM in Chicago.",
    "Okay, loop, book a haircut on Friday.": "book a haircut on Friday.",
    "Okay loop what is on my calendar": "what is on my calendar",
    "... tomorrow at 3. Loop. Book lunch with Sam.": "Book lunch with Sam.",
    "Lupe, move my dentist appointment to 4 PM.": "move my dentist appointment to 4 PM.",
    "I like to loop videos.": None,
    "Look at this.": None,
    "Loop.": "",
}
for text, expected in cases.items():
    got = w.find(text)
    print(f"{'ok ' if got == expected else 'BAD'} {text!r:66} -> {got!r}")
    assert got == expected

ok  'Loop, book a dentist appointment tomorrow at 3 PM in Chicago.'    -> 'book a dentist appointment tomorrow at 3 PM in Chicago.'
ok  'Okay, loop, book a haircut on Friday.'                            -> 'book a haircut on Friday.'
ok  'Okay loop what is on my calendar'                                 -> 'what is on my calendar'
ok  '... tomorrow at 3. Loop. Book lunch with Sam.'                    -> 'Book lunch with Sam.'
ok  'Lupe, move my dentist appointment to 4 PM.'                       -> 'move my dentist appointment to 4 PM.'
ok  'I like to loop videos.'                                           -> None
ok  'Look at this.'                                                    -> None
ok  'Loop.'                                                            -> ''


### Cutting the microphone stream into utterances

`Segmenter.feed(frame)` gets 30 ms frames:
1. the first second calibrates the **noise level** (median RMS); the threshold is `noise x 3` (never below `min_threshold`), and it follows slow changes in room noise;
2. speech starts at the first frame above the threshold, and the last 0.35 s of audio (the **pre-roll**) is put in front so the first syllable survives;
3. **0.6 s** of quiet ends the utterance; anything with less than 0.3 s of loud audio is dropped as a click.

`Transcriber` wraps faster-whisper (or mlx-whisper). Its `initial_prompt` contains "Loop", which makes Whisper spell the name right far more often. Whisper also "hears" *Thank you.* in noise; the assistant drops those.

In [26]:
%%writefile va/speech_in.py
"""Speech in: microphone capture, utterance segmentation by volume, and local speech to text."""
from __future__ import annotations

import collections
import logging
import queue
import time

import numpy as np

log = logging.getLogger("va.stt")


class Segmenter:
    """Cut a stream of audio frames into utterances using the room's noise level.

    feed(frame) returns a finished utterance (float32 array) or None. Pure numpy, so it can be
    tested without a microphone.
    """

    def __init__(self, cfg: dict):
        a = cfg["audio"]
        self.sr = a["sample_rate"]
        self.frame_s = a["frame_ms"] / 1000
        self.factor, self.min_threshold = a["threshold_factor"], a["min_threshold"]
        self.calib_frames = int(a["calibration_seconds"] / self.frame_s)
        self.preroll_frames = int(a["preroll_seconds"] / self.frame_s)
        self.end_frames = int(a["end_silence_seconds"] / self.frame_s)
        self.min_speech_frames = int(a["min_speech_seconds"] / self.frame_s)
        self.max_frames = int(a["max_utterance_seconds"] / self.frame_s)
        self.noise: float | None = None
        self._calib: list[float] = []
        self.reset()

    @property
    def threshold(self) -> float:
        return max((self.noise or 0.0) * self.factor, self.min_threshold)

    @property
    def calibrated(self) -> bool:
        return self.noise is not None

    def reset(self) -> None:
        """Drop any partial utterance (used when the mic is muted). Keeps the noise level."""
        self.preroll: collections.deque = collections.deque(maxlen=max(self.preroll_frames, 1))
        self.frames: list[np.ndarray] = []
        self.in_speech, self.silent_run, self.voiced = False, 0, 0

    def feed(self, frame: np.ndarray) -> np.ndarray | None:
        frame = np.asarray(frame, dtype=np.float32).reshape(-1)
        rms = float(np.sqrt(np.mean(frame ** 2))) if frame.size else 0.0
        if self.noise is None:                                   # 1) calibrate on the first second
            self._calib.append(rms)
            if len(self._calib) >= self.calib_frames:
                self.noise = float(np.median(self._calib))
                log.info("noise level %.4f -> threshold %.4f", self.noise, self.threshold)
            return None
        loud = rms > self.threshold
        if not self.in_speech:
            if loud:                                             # 2) speech starts: keep the pre-roll
                self.in_speech, self.frames = True, list(self.preroll) + [frame]
                self.voiced, self.silent_run = 1, 0
            else:
                self.preroll.append(frame)
                self.noise = 0.98 * self.noise + 0.02 * rms      # follow slow changes in room noise
            return None
        self.frames.append(frame)
        if loud:
            self.voiced += 1
            self.silent_run = 0
        else:
            self.silent_run += 1
        if self.silent_run >= self.end_frames or len(self.frames) >= self.max_frames:   # 3) ~0.6 s quiet ends it
            audio, voiced = np.concatenate(self.frames), self.voiced
            self.reset()
            if voiced >= self.min_speech_frames:
                return audio
            log.debug("dropped a %.2fs blip", voiced * self.frame_s)
        return None


    def flush(self) -> np.ndarray | None:
        """End of a recording: return the utterance in progress, if there is one."""
        audio, voiced = (np.concatenate(self.frames) if self.frames else None), self.voiced
        self.reset()
        return audio if audio is not None and voiced >= self.min_speech_frames else None

    def split(self, audio: np.ndarray) -> list[np.ndarray]:
        """Cut a whole recording into utterances (used for clips recorded in the browser)."""
        n = int(self.sr * self.frame_s)
        out = [u for i in range(0, len(audio) - n + 1, n) if (u := self.feed(audio[i:i + n])) is not None]
        last = self.flush()
        return out + ([last] if last is not None else [])


class Microphone:
    """Continuous capture with sounddevice. Frames go into a queue read by the listener thread."""

    def __init__(self, cfg: dict):
        import sounddevice as sd
        a = cfg["audio"]
        self.sr, self.blocksize = a["sample_rate"], int(a["sample_rate"] * a["frame_ms"] / 1000)
        self.q: queue.Queue = queue.Queue()
        self.stream = sd.InputStream(samplerate=self.sr, channels=1, dtype="float32", blocksize=self.blocksize,
                                     device=a.get("input_device"), callback=self._callback)

    def _callback(self, indata, frames, time_info, status):
        if status:
            log.debug("mic status: %s", status)
        self.q.put(indata[:, 0].copy())

    def start(self):
        self.stream.start()
        return self

    def stop(self):
        self.stream.stop()
        self.stream.close()

    def read(self, timeout: float = 0.5) -> np.ndarray | None:
        try:
            return self.q.get(timeout=timeout)
        except queue.Empty:
            return None

    def drain(self) -> int:
        """Throw away everything recorded so far (e.g. while the assistant was speaking)."""
        n = 0
        while True:
            try:
                self.q.get_nowait()
                n += 1
            except queue.Empty:
                return n


class Transcriber:
    """Local speech to text: faster-whisper anywhere, or mlx-whisper on Apple Silicon."""

    def __init__(self, cfg: dict):
        s = cfg["stt"]
        self.backend, self.model_name = s["backend"], s["model"]
        self.prompt = s.get("initial_prompt")
        self.sr = cfg["audio"]["sample_rate"]
        if self.backend == "faster-whisper":
            from faster_whisper import WhisperModel
            self.model = WhisperModel(self.model_name, device="auto", compute_type=s.get("compute_type", "int8"))
        elif self.backend == "mlx-whisper":
            import mlx_whisper
            self.model = mlx_whisper
        else:
            raise ValueError(f"unknown stt backend {self.backend!r}")
        self.transcribe(np.zeros(self.sr // 2, dtype=np.float32))   # warm up

    def transcribe(self, audio: np.ndarray) -> str:
        audio = np.asarray(audio, dtype=np.float32)
        t0 = time.perf_counter()
        if self.backend == "faster-whisper":
            segments, _ = self.model.transcribe(audio, language="en", beam_size=1, initial_prompt=self.prompt,
                                                condition_on_previous_text=False, vad_filter=False)
            text = " ".join(s.text for s in segments)
        else:
            out = self.model.transcribe(audio, path_or_hf_repo=self.model_name, language="en",
                                        initial_prompt=self.prompt, condition_on_previous_text=False)
            text = out["text"]
        text = text.strip()
        log.info("stt %.2fs (%.1fs audio): %r", time.perf_counter() - t0, len(audio) / self.sr, text)
        return text


def beep(cfg: dict, freq: float = 880.0, seconds: float = 0.12) -> None:
    """A short tone that says 'I am listening again'."""
    try:
        import sounddevice as sd
        sr = 22050
        t = np.arange(int(sr * seconds)) / sr
        tone = 0.2 * np.sin(2 * np.pi * freq * t) * np.hanning(t.size)
        sd.play(tone.astype(np.float32), sr, blocking=True)
    except Exception:
        log.debug("beep failed", exc_info=True)

Writing va/speech_in.py


In [28]:
import importlib, sys, time
from pathlib import Path
import numpy as np

def fresh(*names):
    # Re-import the project modules after a %%writefile, so the newest code runs.
    importlib.invalidate_caches()   # Drive: Python may not notice a file written a moment ago
    for mod in [m for m in sys.modules if m == "va" or m.startswith("va.") or m == "checks"]:
        del sys.modules[mod]
    mods = [importlib.import_module(n) for n in names]
    return mods[0] if len(mods) == 1 else mods

assert Path("va/speech_in.py").exists(), "run the %%writefile va/speech_in.py cell just above first"

# The segmenter on synthetic audio: 2 s noise, 1.2 s "speech", 0.3 s pause, 0.5 s "speech", noise, a 60 ms click, noise.
config, speech_in = fresh("va.config", "va.speech_in")
cfg = config.load_config()
sr, rng = cfg["audio"]["sample_rate"], np.random.default_rng(0)
noise = lambda s: 0.002 * rng.standard_normal(int(s * sr))
voice = lambda s: 0.1 * np.sin(2 * np.pi * 200 * np.arange(int(s * sr)) / sr) + noise(s)
signal = np.concatenate([noise(2), voice(1.2), noise(0.3), voice(0.5), noise(1), voice(0.06), noise(1)])

seg, frame = speech_in.Segmenter(cfg), int(sr * cfg["audio"]["frame_ms"] / 1000)
for i in range(0, len(signal) - frame, frame):
    if (u := seg.feed(signal[i:i + frame])) is not None:
        print(f"utterance ending at {i / sr:.2f} s, {len(u) / sr:.2f} s long")
print(f"noise level {seg.noise:.4f}, threshold {seg.threshold:.4f}  (one utterance: the short pause is kept, the click is dropped)")

INFO:va.stt:noise level 0.0020 -> threshold 0.0061


utterance ending at 4.59 s, 2.97 s long
noise level 0.0020, threshold 0.0060  (one utterance: the short pause is kept, the click is dropped)


In [31]:
# Live: 1 s of quiet to calibrate, then say "Loop, what is on my calendar?" within 8 seconds.
if IN_COLAB:   # no live stream on Colab: record a clip in the browser and cut it the same way
    clip = fresh("va.colab").record_clip(8.0)
    config, speech_in = fresh("va.config", "va.speech_in")
    seg, stt = speech_in.Segmenter(cfg), speech_in.Transcriber(cfg)
    for u in seg.split(clip):
        print(f"{len(u) / sr:.1f} s ->", repr(stt.transcribe(u)))
else:
    mic = speech_in.Microphone(cfg).start()
    seg, stt = speech_in.Segmenter(cfg), speech_in.Transcriber(cfg)
    import time
    end = time.time() + 9
    try:
        while time.time() < end:
            f = mic.read()
            if f is not None and (u := seg.feed(f)) is not None:
                print(f"{len(u) / sr:.1f} s ->", repr(stt.transcribe(u)))
    finally:
        mic.stop()
    print(f"threshold was {seg.threshold:.4f}. If nothing was cut, lower audio.threshold_factor; if noise triggers it, raise it.")

<IPython.core.display.Javascript object>

2026-11-12 05:50:35,790 WARNING MainThread speech to text on the GPU failed (Library libcublas.so.12 is not found or cannot be loaded); using the CPU
INFO:va.stt:stt 33.43s (0.5s audio): 'You'
INFO:va.stt:noise level 0.0041 -> threshold 0.0124
INFO:va.stt:stt 34.73s (2.0s audio): 'Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar

2.0 s -> 'Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar? Loop, what is on my calendar'


INFO:va.stt:stt 1.60s (4.0s audio): 'Hello, hello, hello. Love you, love you.'


4.0 s -> 'Hello, hello, hello. Love you, love you.'


INFO:va.stt:stt 1.49s (0.7s audio): 'So...'


0.7 s -> 'So...'


### Putting it together

`Assistant` owns three background threads, each started once:

| Thread | Job |
|---|---|
| loader | opens memory, warms up the chat model (`keep_alive` keeps it loaded), loads Whisper, opens the mic; then exits |
| listener | mic frames -> segmenter -> Whisper -> wake word -> queue. **While the assistant thinks or speaks (and 0.35 s after), frames are thrown away and the segmenter is reset**, so it never hears itself |
| worker | runs commands one at a time: agent -> speaker; then opens the **8-second follow-up window**, plays the listening tone, and starts memory extraction in the background |

The page and the CLI only read `snapshot()` and call `submit`, `set_muted` and `stop_speaking`.

In [32]:
%%writefile va/assistant.py
"""The always-on assistant: mic -> STT -> wake word -> agent -> TTS, plus state for the web page.

One Assistant owns three background threads, started once:
  loader   loads the models, then exits
  listener reads the microphone and turns utterances into commands
  worker   runs commands one at a time (spoken or typed)
The web page only reads `snapshot()` and calls submit / set_muted / stop_speaking.
"""
from __future__ import annotations

import itertools
import logging
import queue
import re
import threading
import time
from datetime import datetime

from .agent import Agent
from .appointments import AppointmentStore
from .config import resolve_path, setup_logging
from .speech_out import Speaker
from .wake import WakeWord

log = logging.getLogger("va.assistant")

# Whisper often "hears" these in background noise.
_HALLUCINATIONS = re.compile(r"^\W*(thank you|thanks for watching|you|bye|okay|\.+)\W*$", re.I)


class Assistant:
    def __init__(self, cfg: dict, voice: bool = True, speak: bool = True, model: str | None = None,
                 stt: bool | None = None):
        """voice: listen on the microphone. stt: load speech to text even without a microphone
        (Colab: clips recorded in the browser are passed to handle_audio)."""
        self.cfg, self.voice, self.speak, self.model_name = cfg, voice, speak, model
        self.stt = voice if stt is None else stt
        setup_logging(cfg)
        self.store = AppointmentStore(resolve_path(cfg, cfg["appointments"]["db_path"]))
        self.memory = None
        self.agent: Agent | None = None
        self.speaker = Speaker(cfg, enabled=speak)
        self.wake = WakeWord(cfg)
        self.mic = self.transcriber = None

        self._lock = threading.RLock()
        self._jobs: queue.Queue = queue.Queue()
        self._ids = itertools.count(1)
        self.turns: list[dict] = []
        self.heard: list[dict] = []        # every transcript, including ignored ones
        self.loading, self.busy, self.muted = True, False, False
        self.error: str | None = None
        self.notes: list[str] = []
        self.follow_up_until = 0.0
        self.noise_threshold: float | None = None
        self._started = False

    # -- lifecycle --------------------------------------------------------------
    def start(self) -> "Assistant":
        if self._started:
            return self
        self._started = True
        threading.Thread(target=self._load, name="loader", daemon=True).start()
        return self

    def _load(self) -> None:
        try:
            try:
                from .memory import MemoryStore
                self.memory = MemoryStore(self.cfg)
                self.memory.all()                       # fails fast if Qdrant is not running
            except Exception as exc:
                self.memory = None
                self.notes.append(f"Memory is off: {exc}")
                log.warning("memory disabled: %s", exc)
            self.agent = Agent(self.cfg, self.store, self.memory, model=self.model_name)
            self.agent.warm_up()
            if self.stt or self.voice:
                from .speech_in import Transcriber
                self.transcriber = Transcriber(self.cfg)
            if self.voice:
                from .speech_in import Microphone
                self.mic = Microphone(self.cfg).start()
                threading.Thread(target=self._listen, name="listener", daemon=True).start()
            threading.Thread(target=self._work, name="worker", daemon=True).start()
            log.info("assistant ready (model=%s, voice=%s)", self.agent.model, self.voice)
        except Exception as exc:
            self.error = f"{type(exc).__name__}: {exc}"
            log.exception("start-up failed")
        finally:
            self.loading = False

    def wait_ready(self, timeout: float = 300) -> None:
        end = time.monotonic() + timeout
        while self.loading and time.monotonic() < end:
            time.sleep(0.1)
        if self.error:
            raise RuntimeError(self.error)

    # -- status -----------------------------------------------------------------
    @property
    def status(self) -> str:
        if self.error:
            return "error"
        if self.loading:
            return "loading"
        if self.speaker.speaking:
            return "speaking"
        if self.busy or not self._jobs.empty():
            return "thinking"
        if self.muted:
            return "muted"
        return "listening" if self.voice else "ready"

    def _mic_closed(self) -> bool:
        delay = self.cfg["audio"]["unmute_delay_seconds"]
        return (self.muted or self.busy or self.speaker.speaking
                or time.monotonic() - self.speaker.last_done < delay)

    # -- controls -----------------------------------------------------------------
    def submit(self, text: str, source: str = "typed", heard: str | None = None) -> int:
        turn_id = next(self._ids)
        with self._lock:
            self.turns.append({"id": turn_id, "at": datetime.now().strftime("%H:%M:%S"), "source": source,
                               "heard": heard if heard is not None else text, "command": text,
                               "tools": [], "reply": "", "state": "queued", "seconds": None, "first_audio": None})
        self._jobs.put(turn_id)
        return turn_id

    def set_muted(self, muted: bool) -> None:
        self.muted = muted
        if muted:
            self.speaker.stop()

    def stop_speaking(self) -> None:
        self.speaker.stop()

    def run_text(self, text: str, timeout: float = 300) -> dict:
        """Submit a typed command and wait for its turn to finish (used by the CLI and the checks)."""
        tid = self.submit(text)
        end = time.monotonic() + timeout
        while time.monotonic() < end:
            turn = self.turn(tid)
            if turn["state"] in ("done", "error"):
                return turn
            time.sleep(0.05)
        raise TimeoutError(text)

    def turn(self, turn_id: int) -> dict:
        with self._lock:
            return next(t for t in self.turns if t["id"] == turn_id)

    def snapshot(self) -> dict:
        with self._lock:
            turns = [dict(t, tools=list(t["tools"])) for t in self.turns[-30:]]
            heard = list(self.heard[-20:])
        return {"status": self.status, "turns": turns, "heard": heard, "error": self.error, "notes": list(self.notes),
                "follow_up": max(0.0, self.follow_up_until - time.monotonic()),
                "model": self.agent.model if self.agent else self.cfg["llm"]["model"],
                "threshold": self.noise_threshold}

    # -- worker: one command at a time ---------------------------------------------------
    def _work(self) -> None:
        while True:
            turn_id = self._jobs.get()
            turn = self.turn(turn_id)
            self.busy = True
            turn["state"] = "thinking"
            self.speaker.reset_timer()
            t0 = time.perf_counter()
            try:
                result = self.agent.run(turn["command"], on_text=self.speaker.feed,
                                        on_tool=lambda ev: turn["tools"].append(ev))
                self.speaker.flush()
                turn.update(reply=result.reply, seconds=round(result.seconds, 2), state="done",
                            memories=[m["text"] for m in result.memories])
            except Exception as exc:
                log.exception("turn failed")
                turn.update(reply=f"Error: {exc}", state="error")
                self.speaker.say("Sorry, something went wrong.")
            finally:
                self.busy = False
            self.speaker.wait()
            if self.speaker.first_audio_at:
                turn["first_audio"] = round(self.speaker.first_audio_at - t0, 2)
            self.follow_up_until = time.monotonic() + self.cfg["assistant"]["follow_up_seconds"]
            if self.voice and not self.muted and self._jobs.empty() and self.cfg["audio"].get("start_tone"):
                from .speech_in import beep
                beep(self.cfg)
            # Memory extraction after speaking, in the background, so it never slows a reply.
            if self.memory is not None and self.cfg["memory"].get("extract_in_background") and turn["state"] == "done":
                threading.Thread(target=self._extract, args=(turn,), name="memory", daemon=True).start()

    def _extract(self, turn: dict) -> None:
        stored = [r for r in self.memory.extract_and_store(turn["command"], self.agent.model) if r.get("stored")]
        if stored:
            turn["extracted"] = [r["text"] for r in stored]

    # -- listener: mic -> utterance -> text -> wake word -----------------------------------------
    def _listen(self) -> None:
        from .speech_in import Segmenter
        seg = Segmenter(self.cfg)
        was_closed = False
        while True:
            frame = self.mic.read()
            if frame is None:
                continue
            if self._mic_closed():           # never hear ourselves: drop audio while speaking/thinking/muted
                if not was_closed:
                    seg.reset()
                was_closed = True
                self.mic.drain()
                continue
            if was_closed:
                seg.reset()
                was_closed = False
            audio = seg.feed(frame)
            self.noise_threshold = seg.threshold if seg.calibrated else None
            if audio is None:
                continue
            try:
                self._handle_utterance(self.transcriber.transcribe(audio))
            except Exception:
                log.exception("transcription failed")

    def handle_audio(self, audio, started_at: float | None = None) -> str:
        """Transcribe one utterance recorded elsewhere and act on it like the listener would.

        started_at (time.monotonic()) is when the speech started, for the follow-up window."""
        text = self.transcriber.transcribe(audio)
        self._handle_utterance(text, started_at)
        return text

    def _handle_utterance(self, text: str, started_at: float | None = None) -> None:
        if not text or _HALLUCINATIONS.match(text):
            return
        command = self.wake.find(text)
        in_follow_up = (started_at or time.monotonic()) < self.follow_up_until
        if command is None and in_follow_up:
            command = text
        action = "ignored (no wake word)" if command is None else "wake only" if command == "" else "command"
        with self._lock:
            self.heard.append({"at": datetime.now().strftime("%H:%M:%S"), "text": text, "action": action})
        log.info("heard %r -> %s", text, action)
        if command is None:
            return
        if command == "":
            self.speaker.say("Yes?")
            self.speaker.wait()
            self.follow_up_until = time.monotonic() + self.cfg["assistant"]["follow_up_seconds"]
            return
        self.follow_up_until = 0.0
        self.submit(command, source="voice", heard=text)

Writing va/assistant.py


In [33]:
%%writefile cli.py
"""Terminal front end.

    python cli.py            # type commands, hear the replies
    python cli.py --voice    # always-on: speak to it ("Loop, ..."), Ctrl-C to quit
    python cli.py --quiet    # type commands, print replies only
"""
import argparse
import time

from va.assistant import Assistant
from va.config import load_config

p = argparse.ArgumentParser()
p.add_argument("--voice", action="store_true", help="listen on the microphone")
p.add_argument("--quiet", action="store_true", help="do not speak")
p.add_argument("--model", help="override llm.model from config.yaml")
p.add_argument("--config")
args = p.parse_args()

a = Assistant(load_config(args.config), voice=args.voice, speak=not args.quiet, model=args.model).start()
print("loading models...")
a.wait_ready()
for note in a.notes:
    print("note:", note)
print(f"ready: model={a.agent.model}")

if args.voice:
    print(f'Say "{a.cfg["assistant"]["name"]}, ..." Ctrl-C to quit.')
    seen_heard, seen_turns, last_status = 0, set(), None
    try:
        while True:
            snap = a.snapshot()
            if snap["status"] != last_status:
                print(f"[{snap['status']}]")
                last_status = snap["status"]
            for h in a.heard[seen_heard:]:
                print(f"  heard: {h['text']!r} -> {h['action']}")
            seen_heard = len(a.heard)
            for t in snap["turns"]:
                if t["state"] == "done" and t["id"] not in seen_turns:
                    seen_turns.add(t["id"])
                    for ev in t["tools"]:
                        print(f"  tool: {ev['tool']}({ev['arguments']}) -> ok={ev['result'].get('ok')}")
                    print(f"  reply: {t['reply']}  ({t['seconds']} s)")
            time.sleep(0.1)
    except KeyboardInterrupt:
        pass
else:
    try:
        while True:
            text = input("you> ").strip()
            if not text:
                continue
            if text in ("quit", "exit"):
                break
            t = a.run_text(text)
            for ev in t["tools"]:
                print(f"  tool: {ev['tool']}({ev['arguments']}) -> {ev['result']}")
            print(f"loop> {t['reply']}   ({t['seconds']} s)")
            a.speaker.wait()
    except (KeyboardInterrupt, EOFError):
        pass

Writing cli.py


### Check 3: speak the eight commands

Best in a terminal (the mic permission is then for your terminal app):

```bash
cd Week7/voice-assistant
python cli.py --voice
```

Say "Loop, book a dentist appointment tomorrow at 3 PM in Chicago", then the rest (after a reply you have 8 seconds to answer without the wake word, so "yes" works on its own). At least five of eight must work. Every transcript is printed and logged in `logs/assistant.log`; copy the failed ones into the README table, and add new wake-word spellings to `wake.variants`.

Or run it from here for two minutes (laptop only):

In [34]:
if IN_COLAB:
    print("On Colab use the push-to-talk cell below.")
else:
    config, assistant_mod = fresh("va.config", "va.assistant")
    a = assistant_mod.Assistant(config.load_config(), voice=True, speak=True).start()
    a.wait_ready()
    print("listening for 2 minutes. Say: Loop, book a dentist appointment tomorrow at 3 PM in Chicago")
    seen_heard, seen_turns, last = 0, set(), None
    end = time.time() + 120
    while time.time() < end:
        if a.status != last:
            print(f"[{a.status}]"); last = a.status
        for h in a.heard[seen_heard:]:
            print(f"  heard: {h['text']!r} -> {h['action']}")
        seen_heard = len(a.heard)
        for t in a.turns:
            if t["state"] == "done" and t["id"] not in seen_turns:
                seen_turns.add(t["id"]); print(f"  reply: {t['reply']}")
        time.sleep(0.1)
    a.set_muted(True)
    print(a.store.list())

On Colab use the push-to-talk cell below.


**On Colab: push to talk.** From the clip onwards this is the same pipeline: segmenter, Whisper, wake word, follow-up window, agent and speech. Each round records 6 seconds; start speaking when the green "Listening" box appears. The next clip starts right after the reply, so "yes" falls inside the 8-second follow-up window and needs no wake word. Type `q` to finish.

In [35]:
if not IN_COLAB:
    print("Laptop: use the always-on cell above (or python cli.py --voice).")
else:
    config, assistant_mod, colab = fresh("va.config", "va.assistant", "va.colab")
    a = assistant_mod.Assistant(config.load_config(), voice=False, stt=True, speak=True).start()
    a.wait_ready()
    COMMANDS = ["Loop, book a dentist appointment tomorrow at 3 PM in Chicago", "Loop, book a haircut on Friday at 10 AM",
                "Loop, what is on my calendar?", "Loop, move my dentist appointment to 4 PM",
                "Loop, delete my dentist appointment", "yes", "Loop, book lunch with Sam on Friday at noon at Cafe Roma"]
    voice_log = []
    for say in COMMANDS + [None] * 10:
        if input(f"Enter = record, q = stop. Next to say: {say or '(anything)'!r} ").strip().lower() == "q":
            break
        r = colab.listen_once(a, seconds=6)
        voice_log.append({"said": say, **r})
        print(f"  heard: {r['heard']!r} -> {r['action']}\n  tools: {r['tools']}\n  reply: {r['reply']}")
    import pandas as pd
    display(pd.DataFrame(voice_log))
    print(a.store.list())

2026-11-12 05:52:16,302 WARNING loader     memory disabled: No module named 'va.memory'
2026-11-12 05:52:17,091 WARNING loader     speech to text on the GPU failed (Library libcublas.so.12 is not found or cannot be loaded); using the CPU
INFO:va.stt:stt 36.35s (0.5s audio): "Did you make one of the three ticket pings in the house?  We could if we're here to understand what's in our concert diversity,  What else could we see?  house display and play on your cell phone?"
INFO:va.assistant:assistant ready (model=qwen3:8b, voice=False)


Enter = record, q = stop. Next to say: 'Loop, book a dentist appointment tomorrow at 3 PM in Chicago' 


<IPython.core.display.Javascript object>

INFO:va.stt:noise level 0.0053 -> threshold 0.0160
INFO:va.stt:stt 2.20s (5.6s audio): 'Loop, book the dentist appointment tomorrow at 3pm in Chicago.'
INFO:va.assistant:heard 'Loop, book the dentist appointment tomorrow at 3pm in Chicago.' -> command
INFO:va.agent:tool create_appointment({"date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist"}) -> {"ok": true, "id": 1, "appointment": {"id": 1, "date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist", "description": ""}}


INFO:va.agent:turn model=qwen3:8b 4.00s user='book the dentist appointment tomorrow at 3pm in Chicago.' reply="I've booked your Dentist appointment for tomorrow at 3 PM in Chicago. The ID is 1."


  heard: 'Loop, book the dentist appointment tomorrow at 3pm in Chicago.' -> command
  tools: ['create_appointment']
  reply: I've booked your Dentist appointment for tomorrow at 3 PM in Chicago. The ID is 1.
Enter = record, q = stop. Next to say: 'Loop, book a haircut on Friday at 10 AM' 


... (14 outputs left out)

INFO:va.agent:turn model=qwen3:8b 1.88s user='Okay, thank you so much. Now you can go to sleep.' reply="You're welcome! Have a good night."


  heard: 'Okay, thank you so much. Now you can go to sleep.' -> command
  tools: []
  reply: You're welcome! Have a good night.
Enter = record, q = stop. Next to say: 'yes' q


,said,heard,action,utterances,reply,tools
0,"Loop, book a dentist appointment tomorrow at 3 PM in Chicago","Loop, book the dentist appointment tomorrow at 3pm in Chicago.",command,1,I've booked your Dentist appointment for tomorrow at 3 PM in Chicago. The ID is 1.,[create_appointment]
1,"Loop, book a haircut on Friday at 10 AM","Sorry, I'm sorry.",command,2,I'm here to help. What would you like to do?,[]
2,"Loop, what is on my calendar?",Book me a haircut tomorrow.,command,1,You already have an appointment tomorrow at 3 PM in Chicago. Would you like to reschedule your haircut for another t...,[list_appointments]
3,"Loop, move my dentist appointment to 4 PM","Yes, schedule haircut for 5pm.",command,1,I've updated your appointment. Your Haircut is now scheduled for tomorrow at 5 PM in Chicago. The ID is 1.,[update_appointment]
4,"Loop, delete my dentist appointment","Okay, thank you so much. Now you can go to sleep.",command,1,You're welcome! Have a good night.,[]


[{'id': 1, 'date': '2026-11-13 17:00', 'location': 'Chicago', 'title': 'Haircut', 'description': ''}]


---
## Stage 4: Memory (15 minutes)

- **Store.** One Qdrant collection of short facts ("The user prefers morning meetings."), embedded with `nomic-embed-text` using its `search_document:` / `search_query:` prefixes. A fact within 0.92 cosine of one already stored is skipped.
- **Read.** Before each request the agent searches memory with the user's words and puts the top 4 matches above 0.45 into the system prompt.
- **Write on request.** The `remember` tool.
- **Write in the background.** After the reply has been *spoken*, a thread asks the chat model (JSON mode) for lasting facts in what the user said. It is skipped entirely when the user said nothing about themselves (no "I", "my", "me"), which is most commands, so it costs nothing on "yes" or "what is on my calendar".
- **Appointments stay out.** The extraction prompt says so, and a regex drops any fact with times, weekdays, "tomorrow", "appointment" or "booked".

In [42]:
%%writefile va/memory.py
"""Long-term memory: personal facts, searchable by meaning (Ollama embeddings + Qdrant)."""
from __future__ import annotations

import json
import logging
import re
import threading
import time
import uuid
from contextlib import contextmanager
from datetime import datetime

import ollama
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

from .config import resolve_path

log = logging.getLogger("va.memory")

EXTRACT_PROMPT = """You pull long-term personal facts out of one message a user said to their voice assistant.
Keep only lasting facts about the user: preferences, habits, people in their life, places they go, things they like or dislike.
Do NOT keep appointments, bookings, dates, times of specific events, questions, or commands to the assistant.
Write each fact as a short sentence about "the user", e.g. "The user prefers morning meetings."
Return JSON: {"facts": ["..."]}. Return {"facts": []} if there is nothing worth keeping.

Message: """

# Facts that look like scheduling go to the appointments database, never to memory.
_SCHEDULE = re.compile(
    r"\b(appointment|booked?|booking|schedul\w*|calendar|reschedul\w*|cancel\w*|tomorrow|today|tonight|"
    r"next (week|monday|tuesday|wednesday|thursday|friday|saturday|sunday)|"
    r"on (monday|tuesday|wednesday|thursday|friday|saturday|sunday)|\d{1,2}(:\d\d)?\s*(am|pm|a\.m\.|p\.m\.)|"
    r"\d{4}-\d{2}-\d{2})\b",
    re.I,
)
# Explicit "remember that ..." requests are only refused if they are clearly a booking.
_BOOKING = re.compile(r"\b(appointment|booked?|booking|calendar|\d{4}-\d{2}-\d{2})\b", re.I)
_FIRST_PERSON = re.compile(r"\b(i|i'm|im|i've|i'd|my|me|mine|remember)\b", re.I)


def looks_like_schedule(text: str, strict: bool = True) -> bool:
    return bool((_SCHEDULE if strict else _BOOKING).search(text))


class MemoryStore:
    def __init__(self, cfg: dict):
        self.cfg = cfg
        m, e = cfg["memory"], cfg["embeddings"]
        self.collection = m["collection"]
        self.top_k, self.min_score, self.dup_score = m["top_k"], m["min_score"], m["duplicate_score"]
        self.embed_model = e["model"]
        self.q_prefix, self.d_prefix = e.get("query_prefix", ""), e.get("document_prefix", "")
        self.llm = ollama.Client(host=cfg["llm"]["host"])
        self._lock = threading.RLock()
        self._ready = False
        self._path = resolve_path(cfg, m["qdrant_path"]) if m.get("qdrant_path") else None
        if self._path is None:
            self._server = QdrantClient(url=m["qdrant_url"])
        else:
            self._path.mkdir(parents=True, exist_ok=True)

    @contextmanager
    def db(self):
        """The Qdrant client. An embedded store (qdrant_path) locks its folder, so it is opened only
        for each operation; then the notebook and the Streamlit page can both use it."""
        if self._path is None:
            yield self._server
            return
        with self._lock:
            for attempt in range(50):
                try:
                    client = QdrantClient(path=str(self._path))
                    break
                except RuntimeError as exc:          # another process has it open right now
                    if "already accessed" not in str(exc) or attempt == 49:
                        raise
                    time.sleep(0.1)
            try:
                yield client
            finally:
                client.close()

    # -- vectors ------------------------------------------------------------
    def embed(self, texts: list[str], prefix: str) -> list[list[float]]:
        resp = self.llm.embed(model=self.embed_model, input=[prefix + t for t in texts],
                              keep_alive=self.cfg["llm"].get("keep_alive"))
        return resp.embeddings

    def _ensure_collection(self, q, dim: int) -> None:
        if self._ready and self._path is None:
            return
        if not q.collection_exists(self.collection):
            q.create_collection(self.collection, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
        self._ready = True

    # -- public API ---------------------------------------------------------
    def add(self, text: str, source: str = "user") -> dict:
        """Store one fact. Returns {"stored": bool, ...}; near-duplicates are skipped."""
        text = text.strip()
        if not text:
            return {"stored": False, "reason": "empty"}
        if looks_like_schedule(text, strict=(source != "user")):
            return {"stored": False, "reason": "appointments belong in the calendar, not memory"}
        vec = self.embed([text], self.d_prefix)[0]
        with self._lock, self.db() as q:
            self._ensure_collection(q, len(vec))
            hits = self._query(q, vec, 1)
            if hits and hits[0]["score"] >= self.dup_score:
                return {"stored": False, "reason": "already known", "existing": hits[0]["text"]}
            pid = str(uuid.uuid4())
            q.upsert(self.collection, points=[PointStruct(
                id=pid, vector=vec,
                payload={"text": text, "source": source, "created_at": datetime.now().isoformat(timespec="seconds")})])
        log.info("memory stored (%s): %s", source, text)
        return {"stored": True, "id": pid, "text": text}

    def _query(self, q, vec, k: int) -> list[dict]:
        res = q.query_points(self.collection, query=vec, limit=k, with_payload=True)
        return [{"id": str(p.id), "text": p.payload["text"], "score": float(p.score)} for p in res.points]

    def search(self, query: str, k: int | None = None) -> list[dict]:
        if not query.strip():
            return []
        vec = self.embed([query], self.q_prefix)[0]
        with self.db() as q:
            if not q.collection_exists(self.collection):
                return []
            hits = self._query(q, vec, k or self.top_k)
        return [h for h in hits if h["score"] >= self.min_score]

    def all(self) -> list[dict]:
        with self.db() as q:
            if not q.collection_exists(self.collection):
                return []
            points, _ = q.scroll(self.collection, limit=500, with_payload=True)
        rows = [{"id": str(p.id), **p.payload} for p in points]
        return sorted(rows, key=lambda r: r.get("created_at", ""))

    def delete(self, point_id: str) -> None:
        with self.db() as q:
            q.delete(self.collection, points_selector=[point_id])

    def clear(self) -> None:
        with self.db() as q:
            if q.collection_exists(self.collection):
                q.delete_collection(self.collection)
        self._ready = False

    # -- background fact extraction -----------------------------------------
    def extract_facts(self, user_text: str, model: str) -> list[str]:
        """Ask the chat model for lasting personal facts in what the user said."""
        if not _FIRST_PERSON.search(user_text):
            return []          # "yes", "what is on my calendar"... -> skip the LLM call entirely
        kwargs = dict(model=model, messages=[{"role": "user", "content": EXTRACT_PROMPT + user_text}],
                      format="json", options={"temperature": 0}, keep_alive=self.cfg["llm"].get("keep_alive"))
        try:
            think = self.cfg["llm"].get("think")
            resp = self.llm.chat(**kwargs, **({} if think is None else {"think": think}))
        except ollama.ResponseError as exc:
            if "think" not in str(exc).lower():
                raise
            resp = self.llm.chat(**kwargs)          # this model does not take the think flag
        try:
            facts = json.loads(resp.message.content).get("facts", [])
        except (json.JSONDecodeError, AttributeError):
            log.warning("fact extraction returned non-JSON: %r", resp.message.content[:200])
            return []
        return [f.strip() for f in facts if isinstance(f, str) and f.strip() and not looks_like_schedule(f)]

    def extract_and_store(self, user_text: str, model: str) -> list[dict]:
        try:
            return [self.add(f, source="extracted") for f in self.extract_facts(user_text, model)]
        except Exception:
            log.exception("background memory extraction failed")
            return []

Overwriting va/memory.py


### Check 4: it remembers across a restart

`run_stage4` uses its own collection, says "remember that I prefer morning meetings", throws away the agent, the memory client and the history (the "restart"), builds new ones and asks "what time of day do I prefer for meetings".

In [43]:
config, checks = fresh("va.config", "checks")
checks.run_stage4(config.load_config())

INFO:va.memory:memory stored (user): The user prefers morning meetings
INFO:va.agent:tool remember({"fact": "The user prefers morning meetings"}) -> {"ok": true, "stored": true, "id": "ace105c7-97dc-4c16-b3c2-e3785294059b", "text": "The user prefers morning meetings"}
INFO:va.agent:turn model=qwen3:8b 2.91s user='remember that I prefer morning meetings' reply="I've remembered that you prefer morning meetings. Let me know if there's anything else I can assist with!"


reply 1: I've remembered that you prefer morning meetings. Let me know if there's anything else I can assist with! | tools: ['remember']
memories: ['The user prefers morning meetings']


INFO:va.agent:turn model=qwen3:8b 0.84s user='what time of day do I prefer for meetings' reply='You prefer morning meetings.'


reply 2: You prefer morning meetings. | memory used: ['The user prefers morning meetings']
PASS


True

Then do it for real: say "Loop, remember that I prefer morning meetings", stop the program (Ctrl-C), start it again, and ask "Loop, what time of day do I prefer for meetings?".

---
## Stage 5: The web page (15 minutes)

```bash
streamlit run app.py
```

- `st.cache_resource` creates the `Assistant` (and its threads) **once**. Streamlit reruns the script on every click, but it only reads `snapshot()`, so the listening loop never stops.
- `@st.fragment(run_every=0.5)` redraws the live part twice a second without rerunning the controls.
- **Status banner**: grey loading, green listening (with the follow-up countdown), amber thinking, blue speaking, red muted. A short tone plays when it starts listening again.
- **Conversation**: what it heard, a collapsible list of tool calls with arguments and results, the reply with its timing, and any facts remembered from that turn. An expander shows everything the mic heard, including what was ignored for lack of a wake word.
- **Memory and appointments** on the right, with a delete button for each memory.
- **Controls** in the sidebar: mute, stop speaking, a typed-command box.

Set `audio.enabled: false` in the config to use the page without a microphone (the Colab settings do this).

In [44]:
%%writefile app.py
"""Web page for the voice assistant:  streamlit run app.py

The assistant (and its listening loop) lives in background threads created once by
st.cache_resource. Streamlit reruns this script on every click; the page only reads state.
"""
import json

import pandas as pd
import streamlit as st

from va.assistant import Assistant
from va.config import load_config

st.set_page_config(page_title="Loop voice assistant", page_icon="🎙️", layout="wide")


@st.cache_resource
def get_assistant() -> Assistant:
    cfg = load_config()
    return Assistant(cfg, voice=cfg["audio"].get("enabled", True), speak=True).start()


a = get_assistant()

BANNERS = {   # status -> (colour, text a beginner understands)
    "loading":   ("#6b7280", "⏳ Loading models… not listening yet"),
    "listening": ("#16a34a", "🎙️ Listening. Say “{name}, …”"),
    "thinking":  ("#d97706", "🤔 Thinking…"),
    "speaking":  ("#2563eb", "🔊 Speaking (microphone off)"),
    "muted":     ("#dc2626", "🔇 Muted. Not listening"),
    "ready":     ("#16a34a", "⌨️ Ready (typing only, no microphone)"),
    "error":     ("#991b1b", "⚠️ Error: {error}"),
}

# ---- controls (outside the live fragment) --------------------------------------------
with st.sidebar:
    st.header("Controls")
    muted = st.toggle("Mute microphone", value=a.muted)
    if muted != a.muted:
        a.set_muted(muted)
    if st.button("⏹ Stop speaking", use_container_width=True):
        a.stop_speaking()
    with st.form("typed", clear_on_submit=True):
        typed = st.text_input("Type a command", placeholder="book a dentist appointment tomorrow at 3 pm in Chicago")
        if st.form_submit_button("Send", use_container_width=True) and typed.strip():
            a.submit(typed.strip())
    st.caption(f"Model: `{a.snapshot()['model']}`")


@st.fragment(run_every=0.5)
def live():
    snap = a.snapshot()
    colour, text = BANNERS.get(snap["status"], ("#6b7280", snap["status"]))
    text = text.format(name=a.cfg["assistant"]["name"], error=snap["error"])
    if snap["status"] == "listening" and snap["follow_up"] > 0:
        text += f" (or just speak, no wake word needed for {snap['follow_up']:.0f} s)"
    for note in snap["notes"]:
        st.warning(note)
    st.markdown(f"<div style='background:{colour};color:white;padding:18px 24px;border-radius:12px;"
                f"font-size:1.6rem;font-weight:600'>{text}</div>", unsafe_allow_html=True)

    left, right = st.columns([3, 2], gap="large")
    with left:
        st.subheader("Conversation")
        if not snap["turns"]:
            st.caption("Nothing yet.")
        for t in reversed(snap["turns"]):
            with st.container(border=True):
                icon = "🎙️" if t["source"] == "voice" else "⌨️"
                st.markdown(f"**{icon} {t['at']} heard:** {t['heard']}")
                if t["tools"]:
                    with st.expander(f"{len(t['tools'])} tool call(s)"):
                        for ev in t["tools"]:
                            ok = "✅" if ev["result"].get("ok") else "❌"
                            st.markdown(f"{ok} `{ev['tool']}`")
                            st.code(json.dumps({"arguments": ev["arguments"], "result": ev["result"]}, indent=1), "json")
                if t["state"] in ("queued", "thinking"):
                    st.markdown("*…thinking*")
                else:
                    timing = f" ({t['seconds']} s" + (f", first words {t['first_audio']} s)" if t.get("first_audio") else ")") if t["seconds"] else ""
                    st.markdown(f"**🔊 reply:** {t['reply']}{timing}")
                if t.get("extracted"):
                    st.caption("remembered: " + "; ".join(t["extracted"]))
        if snap["heard"]:
            with st.expander("Everything the microphone heard"):
                st.dataframe(pd.DataFrame(snap["heard"][::-1]), hide_index=True, use_container_width=True)

    with right:
        st.subheader("Appointments")
        appts = a.store.list()
        if appts:
            st.dataframe(pd.DataFrame(appts).set_index("id"), use_container_width=True)
        else:
            st.caption("No appointments.")
        st.subheader("Memory")
        if a.memory is None:
            st.caption("Memory is off (is Qdrant running?).")
        else:
            mems = a.memory.all()
            if not mems:
                st.caption("Nothing remembered yet.")
            for m in mems:
                c1, c2 = st.columns([6, 1])
                c1.markdown(f"• {m['text']}  \n<span style='color:gray;font-size:0.8em'>{m.get('source', '')}, {m.get('created_at', '')}</span>",
                            unsafe_allow_html=True)
                if c2.button("🗑", key=f"del-{m['id']}"):
                    a.memory.delete(m["id"])
                    st.rerun()


live()

Overwriting app.py


In [45]:
# Laptop: run `streamlit run app.py` in a terminal, in the project folder.
# Colab: start Streamlit in the background and open it through Colab's port proxy (typing only: no mic there).
if IN_COLAB:
    Path("logs").mkdir(exist_ok=True)
    subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.headless", "true", "--server.port", "8501"],
                     stdout=open("logs/streamlit.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(8)
    from google.colab.output import serve_kernel_port_as_window
    serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

### Check 5

Open the page, say a command and watch the banner go green (listening), amber (thinking), blue (speaking), then green again with a tone.

---
## Stage 6: Measure it (5 minutes)

Runs the eight Stage 1 checks on every model in `llm.compare_models` and writes the table into the README. The time is the sum of the eight agent turns (model warm-up excluded).

In [46]:
config, checks = fresh("va.config", "checks")
cfg = config.load_config()
results = [checks.run_stage1(cfg, model, verbose=False) for model in cfg["llm"]["compare_models"]]
print(checks.write_readme_table(results))
for r in results:
    failed = [row for row in r["rows"] if not row["pass"]]
    for row in failed:
        print(f"\n{r['model']} failed #{row['#']} {row['typed']!r}: tools={row['tools']} reply={row['reply']!r}\n  {row['detail']}")

INFO:va.agent:tool create_appointment({"date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist"}) -> {"ok": true, "id": 1, "appointment": {"id": 1, "date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist", "description": ""}}
INFO:va.agent:turn model=qwen3:8b 3.69s user='book a dentist appointment tomorrow at 3 pm in Chicago' reply='Your dentist appointment is booked for tomorrow at 3 PM in Chicago with ID 1.'
INFO:va.agent:tool create_appointment({"title": "Haircut", "date": "2026-11-13 10:00", "location": "/no_think"}) -> {"ok": false, "error": "The user never said the location '/no_think'. Ask the user where it is."}
INFO:va.agent:turn model=qwen3:8b 2.30s user='book a haircut on Friday at 10 am' reply='Where is your haircut appointment?'
INFO:va.agent:tool list_appointments({}) -> {"ok": true, "count": 1, "appointments": [{"id": 1, "date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist", "description": ""}]}
INFO:va.agent:turn model=qwen3:8b 

| Model | Checks passed (of 8) | Total seconds for the eight checks |
|---|---|---|
| qwen3:8b | 8 | 17.7 |
| llama3.1:8b | 7 | 17.8 |

llama3.1:8b failed #5 'move my dentist appointment to 4 pm': tools=list_appointments, update_appointment! reply='You need to specify the location when moving the appointment. Where is the dentist appointment?'
  rows=[{'id': 1, 'date': '2026-11-13 15:00', 'location': 'Chicago', 'title': 'Dentist appointment', 'description': ''}]


In [47]:
# The same comparison with the guards off: how much of the score comes from code rather than the model?
unguarded = [checks.run_stage1(config.with_overrides(cfg, **{"llm.guards": False}), m, verbose=False) for m in cfg["llm"]["compare_models"]]
pd.DataFrame([{"model": g["model"], "with guards": g["passed"], "without guards": u["passed"],
               "seconds (guards)": g["seconds"], "seconds (no guards)": u["seconds"]} for g, u in zip(results, unguarded)])

INFO:va.agent:tool create_appointment({"date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist"}) -> {"ok": true, "id": 1, "appointment": {"id": 1, "date": "2026-11-13 15:00", "location": "Chicago", "title": "Dentist", "description": ""}}
INFO:va.agent:turn model=qwen3:8b 2.75s user='book a dentist appointment tomorrow at 3 pm in Chicago' reply='Your dentist appointment is booked for tomorrow at 3 PM in Chicago with ID 1.'
INFO:va.agent:tool create_appointment({"date": "2026-11-13 10:00", "location": "/no_think", "title": "Haircut"}) -> {"ok": true, "id": 2, "appointment": {"id": 2, "date": "2026-11-13 10:00", "location": "/no_think", "title": "Haircut", "description": ""}}
INFO:va.agent:turn model=qwen3:8b 2.75s user='book a haircut on Friday at 10 am' reply='Your haircut is booked for Friday at 10 AM in /no_think with ID 2.'
INFO:va.agent:tool list_appointments({}) -> {"ok": true, "count": 2, "appointments": [{"id": 2, "date": "2026-11-13 10:00", "location": "/no_think",

,model,with guards,without guards,seconds (guards),seconds (no guards)
0,qwen3:8b,8,6,17.7,15.8
1,llama3.1:8b,7,7,17.8,15.5


The biggest source of mistakes was the model ignoring the prompt’s rules: with the code checks turned off, qwen3:8b made up a location for the haircut (it put its own control token “/no_think” in the location field) and deleted the dentist appointment without asking first, so it passed only 6 of 8, while llama3.1:8b failed the “move” check because it called update_appointment with the argument id instead of appointment_id. To fix it I moved those rules from the prompt into the tools themselves, so create_appointment refuses a location the user never said and delete_appointment refuses until the user has answered yes, which brought qwen3:8b to 8 of 8 in 17.7 seconds